# Tensor API MxFP4 高性能矩阵算子开发

## 概述

本课程介绍如何使用 Tensor API 开发 MxFP4（Microscaling FP4）高性能矩阵乘法算子。MxFP4 是一种基于微缩放（Microscaling）的低比特浮点格式，通过 fp4 数据配合 fp8 缩放因子，在大幅降低显存带宽占用的同时保持较高的计算精度，是大模型推理与训练场景中的关键加速技术。

本节直接给出最终高性能 kernel 实现，然后详细讲解其中用到的各项优化手段（多核切分、Mutex slot-lock 双缓冲流水、大包搬运、块级同步）的原理和具体效果，最后基于 msopprof 采集的样例性能数据完成性能分析。

### 前置要求

- 已完成本课程第 4 章 **04.03 Tensor API 矩阵算子优化实践**，掌握 Tensor API 基本数据通路（GM → L1 → L0A/L0B → Mmad → L0C → GM）、Layout/Slice/Atom 概念。
- 了解 Cube 核矩阵计算各级存储（GM、L1、L0A/L0B、L0C）的层次关系。
- 已配置 CANN 开发环境，并能访问 Ascend 950PR/DT 设备。

### 学习目标

完成本小节后，开发者应能够：

1. 理解 MxFP4（fp4x2_e1m2_t）和 fp8_e8m0_t 数据类型的存储格式与计算语义；
2. 掌握四路输入（A/ScaleA/B/ScaleB）的数据排布与 Scale 专用 Layout（scalea_nd_layout_ptn/scaleb_dn_layout_ptn）；
3. 掌握 MX 模式 Mmad 的 Trait 配置与调用方式；
4. 理解多核切分、Mutex 双缓冲流水、大包搬运、块级同步等优化手段的原理与效果；
5. 能解读 msopprof 性能指标并进行 Cube/MTE2 理论性能分析；
6. 能独立实现 copy_l0c_to_gm 带 quant 参数的随路量化输出。

### 本节内容

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">章节</th>
      <th align="left">内容</th>
      <th align="left">学习期望</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><a href="#1-算子接口分析">1. 算子接口分析</a></td>
      <td>MxFP4 算子接口、四路输入约定、MX 数据类型与精度校验策略</td>
      <td>明确 MxFP4 算子的数据规格、四路输入和校验方式</td>
    </tr>
    <tr>
      <td><a href="#2-样例工程结构">2. 样例工程结构</a></td>
      <td>工程目录、CMake、Host/Device 代码和 Python 脚本</td>
      <td>掌握 MxFP4 Tensor API 算子工程的完整结构</td>
    </tr>
    <tr>
      <td><a href="#3-核心概念mx-数据类型与四路输入">3. 核心概念：MX 数据类型与四路输入</a></td>
      <td>fp4x2_e1m2_t/fp8_e8m0_t 存储格式、Scale 专用 Layout、L0 Scale 共享存储、MX Mmad</td>
      <td>理解 MX 数据类型存储与四路输入的数据排布方式</td>
    </tr>
    <tr>
      <td><a href="#4-高性能-kernel-实现">4. 高性能 kernel 实现</a></td>
      <td>完整高性能 kernel 源码、编译运行与精度校验</td>
      <td>掌握高性能 MxFP4 kernel 的完整实现</td>
    </tr>
    <tr>
      <td><a href="#5-优化手段详解">5. 优化手段详解</a></td>
      <td>多核切分、Mutex 双缓冲 Ping-Pong、大包搬运、块级同步、F32→BF16 Fixpipe 的原理与效果</td>
      <td>理解每项优化手段的原理及其对性能的贡献</td>
    </tr>
    <tr>
      <td><a href="#6-性能分析">6. 性能分析</a></td>
      <td>msopprof 工具介绍、样例实测数据解读、Cube/MTE2 理论性能分析、L2Cache 优化讨论</td>
      <td>能使用 profiling 工具分析瓶颈并评估优化收益</td>
    </tr>
    <tr>
      <td><a href="#7-课后练习与课后实践">7. 课后练习与课后实践</a></td>
      <td>概念练习 + copy_l0c_to_gm 带 quant 随路量化输出编程实践</td>
      <td>能独立实现带 quant 参数的随路量化输出</td>
    </tr>
  </tbody>
</table>

### 运行环境与硬件说明

- 本节样例 **仅支持 Ascend 950PR/DT**，请在 CANNLab 950 尝鲜体验环境中运行。
- learning-hub 的 notebook 在线体验环境和 CANNLab 910B/910C 云开发环境 **不支持** 本节样例。
- **CANN 版本要求：不低于 9.2.0**。

---
# 1. 算子接口分析

本节实现的 MxFP4 矩阵乘法算子基于微缩放（Microscaling）数据格式，其计算公式为：

```text
C = (ScaleA ⊗ A) × (ScaleB ⊗ B)
```

其中 `⊗` 表示 scale 沿 K 方向广播：每 32 个矩阵元素共享一个 `fp8_e8m0_t` 缩放因子。MxFP4 算子有 **四路输入**（A、ScaleA、B、ScaleB），并在 L0C 中使用 `float` 累加，最终通过 Fixpipe 转换为 `bfloat16_t` 写回 GM。

![MxMatmul](images/07_08_tensor_api_mxfp4_matmul/mx_matmul_concept.png)

## 1.1 MX 数据类型

MxFP4 涉及三种 MX 数据类型：

| 数据类型 | 位宽 | 结构 | 说明 |
| --- | --- | --- | --- |
| `fp4x2_e1m2_t` | 4-bit | 1 符号位 + 1 指数位 + 2 尾数位 | 两个 fp4 打包为 1 byte，sizeof=0.5B |
| `fp8_e8m0_t` | 8-bit | 8 指数位、0 尾数位 | 表示 `2^(value-127)`，value=127 时缩放因子为 1 |
| `bfloat16_t` | 16-bit | 1 符号位 + 8 指数位 + 7 尾数位 | 输出类型，L0C float 经 Fixpipe 转换 |

## 1.2 输入输出规格

| 参数 | 取值 | 说明 |
| --- | --- | --- |
| M | 8192 | 矩阵 M 维度 |
| N | 8192 | 矩阵 N 维度 |
| K | 8192 | 矩阵 K 维度 |
| A `[M, K]` | `fp4x2_e1m2_t` ND | 左矩阵，行主序，2 元素打包 1 byte |
| ScaleA `[M, scale_k]` | `fp8_e8m0_t` ND | A 的缩放因子，K 方向每 32 元素共享 1 个 scale |
| B `[N, K]` | `fp4x2_e1m2_t` DN | 右矩阵，按 `[N, K]` 形式输入 kernel（物理转置存储） |
| ScaleB `[N, scale_k]` | `fp8_e8m0_t` DN | B 的缩放因子，推荐按 `[N, scale_k]` 输入（ND 形式需 K 方向 2Byte 连续对齐） |
| C `[M, N]` | `bfloat16_t` ND | 输出矩阵 |

其中 `scale_k = align_even(ceil(K/32)) = 256`：先对 K 除以 32 向上取整，再对齐到 2 的倍数（硬件要求 scale 数据在 K 方向满足 2Byte 连续对齐）。

## 1.3 关键参数

| 参数 | 值 | 说明 |
| --- | --- | --- |
| base_m = base_n = base_k | 256 | Cube 计算基本块大小 |
| single_core_m | 2048 | 单核 M 方向计算范围 |
| single_core_n | 1024 | 单核 N 方向计算范围 |
| single_core_k | 8192 | 单核 K 方向计算范围 |
| step_k | 2 | A/B 在 K 方向的大包搬运步长 |
| scale_factor_k | 4 | Scale 相对 A/B 的 K 方向搬运比例 |
| 核数 | `32` | 使用核数 |

> **约束**：本样例要求 K 能被 base_k 整除（不支持 K 方向尾块）。

## 1.4 精度校验策略

输出为 `bfloat16_t`，与 Python numpy 计算的 golden 进行对比。容差设置：`rtol=1e-3`、`atol=1e-3`、误差比例阈值 `1e-3`。

---
# 2. 样例工程结构

### 目录结构介绍

本课程代码按照如下目录结构存放，所有样例均位于 `src/07_08_tensor_api_mxfp4_matmul`。

```text
src/07_08_tensor_api_mxfp4_matmul/
├── scripts/
│   ├── gen_data.py                        # 输入数据生成脚本（MxFP4）
│   ├── gen_data_quant.py                  # 实践题数据生成脚本（含量化参数）
│   ├── verify_result.py                   # 输出结果校验脚本（BF16）
│   └── verify_result_quant.py             # 实践题校验脚本（INT8）
├── CMakeLists.txt                         # 编译工程文件（3 个 target）
├── run.sh                                 # 编译运行脚本
├── data_utils.h                           # 数据读入写出工具函数
├── mmad_mx_host.h                         # Host 侧启动函数模板
├── mmad_mx_high_performance_kernel.h      # 高性能 kernel
├── mmad_mx_high_performance.asc           # 高性能 kernel 入口
├── mmad_mx_quant_practice_kernel.h        # 实践题 TODO 模板 kernel
├── mmad_mx_quant_practice.asc             # 实践题 TODO 模板入口
├── mmad_mx_quant_answer_kernel.h          # 实践题参考实现 kernel
└── mmad_mx_quant_answer.asc             # 实践题参考实现入口
```

### 依赖安装

本样例依赖 `ml_dtypes==0.2.0`（bfloat16 支持）和 `en_dtypes==0.0.4`（float4_e1m2 支持），版本需与官方样例保持一致。

In [ ]:
!pip install ml_dtypes==0.2.0 en_dtypes==0.0.4

### 2.1 CMakeLists.txt 和 run.sh

In [ ]:
%%writefile src/07_08_tensor_api_mxfp4_matmul/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_RUN_MODE "npu" CACHE STRING "Run mode: npu")
set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "Tensor API only supports dav-3510 in this sample")

if(NOT CMAKE_ASC_ARCHITECTURES STREQUAL "dav-3510")
    message(FATAL_ERROR "mmad_mx only supports CMAKE_ASC_ARCHITECTURES=dav-3510")
endif()

find_package(ASC REQUIRED)

project(mmad_mx LANGUAGES ASC CXX)

set(ASCEND_HOME $ENV{ASCEND_HOME_PATH})

set(ASC_INCLUDE_DIR "${ASCEND_HOME}/asc/")
include_directories(
    "${ASC_INCLUDE_DIR}"
)

foreach(target_name IN ITEMS
    mmad_mx_high_performance
    mmad_mx_quant_practice
    mmad_mx_quant_answer
)
    add_executable(${target_name} ${target_name}.asc)
    target_compile_options(${target_name} PRIVATE
        $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
    )
endforeach()


In [ ]:
%%writefile src/07_08_tensor_api_mxfp4_matmul/run.sh
#!/usr/bin/env bash
# CANN 版本要求：不低于 9.2.0（Tensor API asc::te:: 接口自 9.2.0 起提供）
set -euo pipefail

npu_arch="dav-3510"
case_name="all"

for arg in "$@"; do
    case "$arg" in
        --npu-arch=*)
            npu_arch="${arg#*=}"
            ;;
        --case=*)
            case_name="${arg#*=}"
            ;;
        -h|--help)
            echo "Usage: bash run.sh [--npu-arch=dav-3510] [--case=high_performance|quant_practice|quant_answer|all]"
            exit 0
            ;;
    esac
done

if [ "$npu_arch" != "dav-3510" ]; then
    echo "Tensor API MxFP4 sample only supports --npu-arch=dav-3510"
    exit 1
fi

if [ -z "${ASCEND_HOME_PATH:-}" ]; then
    echo "Please set ASCEND_HOME_PATH before building."
    exit 1
fi

if [ -f "${ASCEND_HOME_PATH}/set_env.sh" ]; then
    # shellcheck disable=SC1091
    source "${ASCEND_HOME_PATH}/set_env.sh"
fi

rm -rf build_out input output
cmake -S . -B build_out -DCMAKE_ASC_ARCHITECTURES="$npu_arch"
mkdir -p output

run_one() {
    local name="$1"
    local binary="$2"
    local gen_script="$3"
    local verify_script="$4"
    local out_file="output/${name}.bin"
    echo "[GEN] ${gen_script}"
    python3 "scripts/${gen_script}"
    echo "[BUILD] ${binary}"
    cmake --build build_out -j"$(nproc)" --target "${binary}"
    echo "[RUN] ${name}"
    ./build_out/${binary}
    echo "[VERIFY] ${name}"
    python3 "scripts/${verify_script}" "${out_file}"
}

case "$case_name" in
    high_performance)
        run_one high_performance mmad_mx_high_performance gen_data.py verify_result.py
        ;;
    quant_practice)
        run_one quant_practice mmad_mx_quant_practice gen_data_quant.py verify_result_quant.py
        ;;
    quant_answer)
        run_one quant_answer mmad_mx_quant_answer gen_data_quant.py verify_result_quant.py
        ;;
    all)
        run_one high_performance mmad_mx_high_performance gen_data.py verify_result.py
        ;;
    *)
        echo "Unsupported case: ${case_name}"
        echo "Use high_performance, quant_practice, quant_answer, or all."
        exit 1
        ;;
esac


### 2.2 Host 侧公共代码和数据读写工具

In [ ]:
%%writefile src/07_08_tensor_api_mxfp4_matmul/data_utils.h
#ifndef MMAD_MX_DATA_UTILS_H
#define MMAD_MX_DATA_UTILS_H

#include <fcntl.h>
#include <sys/stat.h>
#include <unistd.h>

#include <cstdio>
#include <fstream>
#include <string>

#define ERROR_LOG(fmt, args...) fprintf(stdout, "[ERROR] " fmt "\n", ##args)

inline bool ReadFile(const std::string &filePath, size_t &fileSize, void *buffer, size_t bufferSize)
{
    struct stat sBuf;
    int fileStatus = stat(filePath.data(), &sBuf);
    if (fileStatus == -1) {
        ERROR_LOG("failed to get file: %s", filePath.c_str());
        return false;
    }
    if (S_ISREG(sBuf.st_mode) == 0) {
        ERROR_LOG("%s is not a file", filePath.c_str());
        return false;
    }

    std::ifstream file(filePath, std::ios::binary);
    if (!file.is_open()) {
        ERROR_LOG("open file failed: %s", filePath.c_str());
        return false;
    }

    std::filebuf *buf = file.rdbuf();
    size_t size = buf->pubseekoff(0, std::ios::end, std::ios::in);
    if (size == 0) {
        ERROR_LOG("file size is 0: %s", filePath.c_str());
        return false;
    }
    if (size > bufferSize) {
        ERROR_LOG("file size is larger than buffer size: %s", filePath.c_str());
        return false;
    }

    buf->pubseekpos(0, std::ios::in);
    buf->sgetn(static_cast<char *>(buffer), size);
    fileSize = size;
    return true;
}

inline bool WriteFile(const std::string &filePath, const void *buffer, size_t size)
{
    if (buffer == nullptr) {
        ERROR_LOG("write file failed, buffer is nullptr");
        return false;
    }

    int fd = open(filePath.c_str(), O_RDWR | O_CREAT | O_TRUNC, S_IRUSR | S_IWRITE);
    if (fd < 0) {
        ERROR_LOG("open file failed: %s", filePath.c_str());
        return false;
    }

    size_t writeSize = write(fd, buffer, size);
    (void)close(fd);
    if (writeSize != size) {
        ERROR_LOG("write file failed: %s", filePath.c_str());
        return false;
    }
    return true;
}

#endif


In [ ]:
%%writefile src/07_08_tensor_api_mxfp4_matmul/mmad_mx_host.h
#ifndef MMAD_MX_HOST_H
#define MMAD_MX_HOST_H

#include "acl/acl.h"
#include "data_utils.h"

#include <cstdint>
#include <cstdio>
#include <string>

#define CHECK_ACL_RET(expr)                                      \
    do {                                                         \
        aclError ret = (expr);                                   \
        if (ret != ACL_SUCCESS) {                                \
            ERROR_LOG("%s failed, ret = %d", #expr, ret);        \
            return 1;                                            \
        }                                                        \
    } while (0)

template <typename LaunchFunc>
int run_mmadmx_host(
    size_t a_file_size, size_t b_file_size, size_t as_file_size, size_t bs_file_size, size_t c_file_size,
    const char *output_path, LaunchFunc launch)
{
    CHECK_ACL_RET(aclInit(nullptr));
    int32_t device_id = 0;
    CHECK_ACL_RET(aclrtSetDevice(device_id));

    aclrtStream stream = nullptr;
    CHECK_ACL_RET(aclrtCreateStream(&stream));

    uint8_t *a_host = nullptr, *b_host = nullptr, *as_host = nullptr, *bs_host = nullptr, *c_host = nullptr;
    uint8_t *a_device = nullptr, *b_device = nullptr, *as_device = nullptr, *bs_device = nullptr, *c_device = nullptr;

    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&a_host), a_file_size));
    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&b_host), b_file_size));
    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&as_host), as_file_size));
    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&bs_host), bs_file_size));
    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&c_host), c_file_size));

    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&a_device), a_file_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&b_device), b_file_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&as_device), as_file_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&bs_device), bs_file_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&c_device), c_file_size, ACL_MEM_MALLOC_HUGE_FIRST));

    size_t file_size = a_file_size;
    if (!ReadFile("./input/x1_gm.bin", file_size, a_host, a_file_size)) { return 1; }
    file_size = b_file_size;
    if (!ReadFile("./input/x2_gm.bin", file_size, b_host, b_file_size)) { return 1; }
    file_size = as_file_size;
    if (!ReadFile("./input/x1_scale_gm.bin", file_size, as_host, as_file_size)) { return 1; }
    file_size = bs_file_size;
    if (!ReadFile("./input/x2_scale_gm.bin", file_size, bs_host, bs_file_size)) { return 1; }

    CHECK_ACL_RET(aclrtMemcpy(a_device, a_file_size, a_host, a_file_size, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL_RET(aclrtMemcpy(b_device, b_file_size, b_host, b_file_size, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL_RET(aclrtMemcpy(as_device, as_file_size, as_host, as_file_size, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL_RET(aclrtMemcpy(bs_device, bs_file_size, bs_host, bs_file_size, ACL_MEMCPY_HOST_TO_DEVICE));

    launch(a_device, b_device, as_device, bs_device, c_device, stream);
    CHECK_ACL_RET(aclrtSynchronizeStream(stream));

    CHECK_ACL_RET(aclrtMemcpy(c_host, c_file_size, c_device, c_file_size, ACL_MEMCPY_DEVICE_TO_HOST));
    if (!WriteFile(output_path, c_host, c_file_size)) { return 1; }

    (void)aclrtFree(a_device); (void)aclrtFree(b_device);
    (void)aclrtFree(as_device); (void)aclrtFree(bs_device);
    (void)aclrtFree(c_device);
    (void)aclrtFreeHost(a_host); (void)aclrtFreeHost(b_host);
    (void)aclrtFreeHost(as_host); (void)aclrtFreeHost(bs_host);
    (void)aclrtFreeHost(c_host);
    (void)aclrtDestroyStream(stream);
    (void)aclrtResetDevice(device_id);
    (void)aclFinalize();
    return 0;
}

#endif


### 2.3 Python 输入生成脚本和精度校验脚本

In [ ]:
%%writefile src/07_08_tensor_api_mxfp4_matmul/scripts/gen_data.py
import os

import numpy as np
import ml_dtypes
import en_dtypes

bfloat16 = ml_dtypes.bfloat16
fp4_e1m2x2 = en_dtypes.float4_e1m2


def pack_two_fp4(scale_matrix):
    scale_matrix_row = scale_matrix.shape[0]
    scale_matrix_col = scale_matrix.shape[1]
    scale_matrix_bin = scale_matrix.flatten()
    scale_matrix_high = scale_matrix_bin[::2].view(np.uint8)
    scale_matrix_low = scale_matrix_bin[1::2].view(np.uint8)
    low_bits = (scale_matrix_low & 0x0F) << 4
    high_bits = scale_matrix_high & 0x0F
    combined = low_bits | high_bits
    scale_matrix_bin = combined.reshape(scale_matrix_row, scale_matrix_col // 2)
    return scale_matrix_bin


def main():
    m, n, k = 8192, 8192, 8192
    sk = (int)(np.ceil(k / 64) * 2)

    os.makedirs("input", exist_ok=True)
    os.makedirs("output", exist_ok=True)

    np.random.seed(42)
    x1_gm = np.random.uniform(-1, 2, [m, k]).astype(fp4_e1m2x2)
    x2_gm = np.random.uniform(-1, 2, [k, n]).astype(fp4_e1m2x2)

    x1_scale_gm = np.random.randint(127, 130, [m, sk]).astype(np.uint8)
    x2_scale_gm = np.random.randint(127, 130, [sk, n]).astype(np.uint8)

    x1_mx = 2 ** (x1_scale_gm.astype(np.float64) - 127)
    x2_mx = 2 ** (x2_scale_gm.astype(np.float64) - 127)
    x1_full = np.zeros([m, k], dtype=np.float64)
    x2_full = np.zeros([k, n], dtype=np.float64)

    for i in range(x1_gm.shape[1]):
        x1_full[:, i] = x1_gm[:, i] * x1_mx[:, i // 32]
        x2_full[i, :] = x2_gm[i, :] * x2_mx[i // 32, :]

    golden = np.matmul(x1_full.astype(np.float64), x2_full.astype(np.float64)).astype(bfloat16)

    x2_gm = x2_gm.transpose()
    x2_scale_gm = x2_scale_gm.transpose()
    x1_gm_packed = pack_two_fp4(x1_gm)
    x2_gm_packed = pack_two_fp4(x2_gm)
    x1_gm_packed.tofile("input/x1_gm.bin")
    x2_gm_packed.tofile("input/x2_gm.bin")
    x1_scale_gm.tofile("input/x1_scale_gm.bin")
    x2_scale_gm.tofile("input/x2_scale_gm.bin")
    golden.tofile("output/golden.bin")

    print(f"generated MxFP4 input data: A[{m},{k}], B[{k},{n}], ScaleA[{m},{sk}], ScaleB[{sk},{n}]")
    print(f"golden output saved to output/golden.bin")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/07_08_tensor_api_mxfp4_matmul/scripts/verify_result.py
import sys

import numpy as np
import ml_dtypes

bfloat16 = ml_dtypes.bfloat16

RELATIVE_TOL = 1e-3
ABSOLUTE_TOL = 1e-3
ERROR_TOL = 1e-3


def main():
    if len(sys.argv) != 2:
        raise SystemExit("Usage: python3 verify_result.py output/<case>.bin")

    output_path = sys.argv[1]
    golden_path = "output/golden.bin"

    output = np.fromfile(output_path, dtype=bfloat16).reshape(-1)
    golden = np.fromfile(golden_path, dtype=bfloat16).reshape(-1)

    if output.size != golden.size:
        raise SystemExit(f"size mismatch: output {output.size}, golden {golden.size}")

    different = ~np.isclose(
        output.astype(np.float32),
        golden.astype(np.float32),
        rtol=RELATIVE_TOL,
        atol=ABSOLUTE_TOL,
        equal_nan=True,
    )
    diff_idx = np.where(different)[0]
    error_count = diff_idx.size
    total = golden.size

    print(f"Total elements compared: {total}")
    print(f"Total error elements: {error_count}")

    if error_count > 0:
        for idx in diff_idx[:100]:
            golden_val = float(golden[idx])
            output_val = float(output[idx])
            denom = abs(golden_val) if abs(golden_val) > 1e-12 else 1.0
            print(f"data index: {idx:06d}, expected: {golden_val:.9f}, actual: {output_val:.9f}, "
                  f"rdiff: {abs(output_val - golden_val) / denom:.6f}")

    error_ratio = float(error_count) / total
    print(f"error ratio: {error_ratio:.4f}, tolerance: {ERROR_TOL:.4f}")

    if error_ratio > ERROR_TOL:
        raise SystemExit("verify failed!")
    print("test pass!")


if __name__ == "__main__":
    main()


---
# 3. 核心概念：MX 数据类型与四路输入

## 3.1 MxMatmul 概念

普通 Matmul 只有两路矩阵输入，计算 `C = A × B`；MxMatmul 在此基础上额外引入两路 ScaleA/ScaleB 输入。A、B 为 MxFP4 低比特数据，ScaleA、ScaleB 为缩放因子，计算时矩阵数据与对应缩放因子共同参与运算：

```text
C = (ScaleA ⊗ A) × (ScaleB ⊗ B)
```

K 方向每 **32** 个矩阵元素共享一个 scale，scale 沿 K 方向广播到对应的一组 MxFP4 数据上，实际计算时硬件在 Mmad 指令中自动将 fp4 数据与对应 scale 相乘后再累加。

![MxMatmul](images/07_08_tensor_api_mxfp4_matmul/mx_matmul_concept.png)

## 3.2 样例规格

本节实现固定 shape 为 `8192 × 8192 × 8192` 的 MxFP4 Matmul，输出数据类型为 `bfloat16_t`：

| 输入/输出 | 逻辑形状 | 数据类型 | 数据排布 | 说明 |
| --- | --- | --- | --- | --- |
| A | `[M, K]` | `fp4x2_e1m2_t` | ND | 左矩阵，每字节打包 2 个 fp4 元素 |
| ScaleA | `[M, scale_k]` | `fp8_e8m0_t` | ND | A 的缩放因子，K 方向每 32 个元素共享一个 |
| B | `[N, K]` | `fp4x2_e1m2_t` | DN | 右矩阵，按 `[N, K]` 形式输入 kernel |
| ScaleB | `[N, scale_k]` | `fp8_e8m0_t` | DN | B 的缩放因子，推荐按 `[N, scale_k]` 输入 |
| C | `[M, N]` | `bfloat16_t` | ND | 输出矩阵 |

其中：

- 两个 `fp4x2_e1m2_t` 元素打包存储在一个字节中，因此 K 需要为**偶数**；
- `scale_k = align_even(ceil(K / 32))`：先向上整除再对齐到 2 的倍数。硬件约束要求 scale 数据在 K 方向满足 2Byte 连续对齐，因此 scale_k 必须为偶数；
- 由于硬件约束，ScaleB 若按 `[scale_k, N]` 输入需要 K 方向 2Byte 连续，因此推荐按 `[N, scale_k]`（DN 排布）输入；
- **本节样例暂不支持 K 方向存在尾块的场景**，即要求 `K` 能被 `baseK` 整除。

## 3.3 关键参数

| 参数 | 值 | 说明 |
| --- | --- | --- |
| `M` | `8192` | 矩阵 M 维度规模 |
| `N` | `8192` | 矩阵 N 维度规模 |
| `K` | `8192` | 矩阵 K 维度规模 |
| `base_m` | `256` | Cube 计算基本块 M 维度大小 |
| `base_k` | `256` | Cube 计算基本块 K 维度大小 |
| `base_n` | `256` | Cube 计算基本块 N 维度大小 |
| `single_core_m` | `2048` | 单核 M 方向计算范围 |
| `single_core_n` | `1024` | 单核 N 方向计算范围 |
| `single_core_k` | `8192` | 单核 K 方向计算范围 |
| `step_k` | `2` | GM→L1 搬运中，A/B 在 K 方向的大包搬运步长 |
| `scale_factor_k` | `4` | GM→L1 搬运中，ScaleA/ScaleB 相对 A/B 的 K 方向搬运比例 |
| 核数 | `32` | 使用核数 |

## 3.4 四路输入排布与 Scale 专用 Layout

与普通 Matmul 相比，MxMatmul 在 kernel 中多了 scale 的搬运、加载和参与计算路径。**A/B 与 ScaleA/ScaleB 必须按相同的 K block 节奏进入计算流水**，否则 Cube 侧拿不到匹配的缩放信息。四路输入每路在 GM 和 L1 中使用不同的 Layout：

| 输入 | GM Layout | L1 Layout | 说明 |
| --- | --- | --- | --- |
| A `[M, K]` | `nd_ext_layout_ptn` | `nz_layout_ptn`（C0=64） | 行主序，fp4x2 打包 |
| B `[K, N]` | `dn_ext_layout_ptn` | `zn_layout_ptn`（C0=64） | 列主序（转置存储），fp4x2 打包 |
| ScaleA `[M, scale_k]` | `scalea_nd_layout_ptn` | `zz_layout_ptn`（C0=2） | Scale 专用行主序 |
| ScaleB `[scale_k, N]` | `scaleb_dn_layout_ptn` | `nn_layout_ptn`（C0=2） | Scale 专用列主序 |

其中 `C0_ELEMENT_B4=64` 是 fp4 数据的分形内层块大小，`C0_ELEMENT_SCALE=2` 是 scale 数据的分形内层块大小。Scale 使用专用 Layout（`scalea_nd_layout_ptn`/`scaleb_dn_layout_ptn`）而非普通的 `nd_layout_ptn`/`dn_layout_ptn`，因为 scale 的数据排布与矩阵数据不同，需要专门的格式转换。

`copy_gm_to_l1` 搬运时会自动完成从 GM Layout 到 L1 Layout 的格式转换。A 从 nd_ext→nz，B 从 dn_ext→zn，ScaleA 从 scalea_nd→zz，ScaleB 从 scaleb_dn→nn。

![四路输入数据格式](images/07_08_tensor_api_mxfp4_matmul/four_path_format.png)

![四路输入搬运与计算](images/07_08_tensor_api_mxfp4_matmul/four_path_input.png)

## 3.5 L0 中 Scale 的特殊存储

以ScaleA为例，下面代码展示何如通过ScaleA在L0A上的地址推导出在L0ScaleA上的地址：

```cpp
// L0A 张量
auto l0_tensor_a = make_tensor(make_mem_ptr(l0_buf_a), l0_layout_a);

// L0ScaleA 张量
auto l0_tensor_as = make_tensor(
    make_mem_ptr<asc::te::location::l0scalea, fp8_e8m0_t>(
        reinterpret_cast<uint64_t>(l0_buf_a) / 16),
    l0_layout_as);
```

`asc::te::location::l0scalea` 指定内存位置为 L0ScaleA，`reinterpret_cast<uint64_t>(l0_buf_a) / 16` 将 L0A 的物理地址转换为 L0ScaleA 的指针偏移。

## 3.6 MX 模式 Mmad

MX 模式的 Mmad 需要配置特殊的 Trait：

```cpp
constexpr mmad_trait MX_MMAD_TRAIT =
    mmad_trait{0, false, false, true, mmad_type::mx};
```

MX 模式的 Trait 设置 `disable_gemv=true`（第 4 个参数）和 `mmad_type::mx`（第 5 个参数）。其余参数含义：第 1 个 `0` 为 `fm_offset`，第 2/3 个 `false` 分别为 `k_direction_align` 和 `init_with_btbuf`。

MX Mmad 的调用方式：

```cpp
mmad(mmad_atom.with(params), l0_c_tensor, l0_a_tensor, l0_b_tensor);
```

硬件会自动读取 L0ScaleA/L0ScaleB 中的缩放因子，将 fp4 数据乘以对应 scale 后再累加到 L0C。`init_with_zero` 的语义：首个 K block 设为 `true` 初始化 L0C 为零，后续设为 `false` 持续累加。


---
# 4. 高性能 kernel 实现

高性能 kernel `kernel_mmadmx` 采用类封装，集成多核切分、Mutex slot-lock 双缓冲流水、大包搬运和块级同步等优化手段，在 Ascend 950PR 上实现 94% 的 Cube 利用率。

整体数据流：

```text
物理地址                               流水
  GM
  |  A:copy_gm_to_l1 / ScaleA:copy_gm_to_l1   MTE2
  |  B:copy_gm_to_l1 / ScaleB:copy_gm_to_l1   MTE2
  v
  L1
  |  A:copy_l1_to_l0a / ScaleA:copy_l1_to_l0scalea        MTE1
  |  B:copy_l1_to_l0b / ScaleB:copy_l1_to_l0scaleb        MTE1
  v
  L0A / L0B / L0ScaleA / L0ScaleB
  |
  |  Mmad                              M
  v
  L0C
  |
  |  copy_l0c_to_gm: float -> bfloat16       FIX
  v
  GM
```

写入 kernel 和入口文件后编译运行验证：

In [ ]:
%%writefile src/07_08_tensor_api_mxfp4_matmul/mmad_mx_high_performance_kernel.h
#ifndef MMAD_MX_HIGH_PERFORMANCE_KERNEL_H
#define MMAD_MX_HIGH_PERFORMANCE_KERNEL_H

#include "c_api/asc_simd.h"
#include "tensor_api/tensor.h"
#include "utils/std/cmath.h"

__aicore__ __inline__ constexpr uint32_t align_even(uint32_t a) { return (a + 1) / 2 * 2; }

constexpr uint32_t SCALE_CEIL_NUMBER = 32;
constexpr uint32_t SCALE_ALIGN_NUMBER = 2;
constexpr uint32_t C0_ELEMENT_SCALE = 2;
constexpr uint32_t C0_ELEMENT_L0C = 16;
constexpr uint32_t C0_ELEMENT_B4 = 64;
constexpr uint32_t CUBE_BLOCK = 16;

template <
    uint32_t M_, uint32_t N_, uint32_t K_, uint32_t single_core_m_, uint32_t single_core_n_, uint32_t single_core_k_,
    uint32_t base_m_, uint32_t base_n_, uint32_t base_k_, uint32_t step_k_, uint32_t scale_factor_k_>
struct kernel_trait {
    static constexpr uint32_t M = M_;
    static constexpr uint32_t N = N_;
    static constexpr uint32_t K = K_;

    static constexpr uint32_t single_core_m = single_core_m_;
    static constexpr uint32_t single_core_n = single_core_n_;
    static constexpr uint32_t single_core_k = single_core_k_;

    static constexpr uint32_t base_m = base_m_;
    static constexpr uint32_t base_n = base_n_;
    static constexpr uint32_t base_k = base_k_;

    static constexpr uint32_t step_k = step_k_;
    static constexpr uint32_t scale_factor_k = scale_factor_k_;
};

constexpr asc::te::mmad_trait MX_MMAD_TRAIT =
    asc::te::mmad_trait{0, false, false, true, asc::te::mmad_type::mx};
struct mmad_trait_mx {
    using trait_type = asc::te::mmad_trait;
    static constexpr const trait_type value = MX_MMAD_TRAIT;
};

template <typename Trait>
class kernel_mmadmx {
public:
    __aicore__ inline kernel_mmadmx() {}

    __aicore__ inline void process(
        __gm__ fp4x2_e1m2_t* a, __gm__ fp4x2_e1m2_t* b, __gm__ fp8_e8m0_t* as, __gm__ fp8_e8m0_t* bs,
        __gm__ bfloat16_t* c)
    {
        init_compute_params();

        // Init GM tensor
        auto gm_tensor_a = asc::te::make_tensor(
            asc::te::make_mem_ptr(a), asc::te::make_frame_layout<asc::te::nd_ext_layout_ptn>(Trait::M, Trait::K));
        auto gm_tensor_b = asc::te::make_tensor(
            asc::te::make_mem_ptr(b), asc::te::make_frame_layout<asc::te::dn_ext_layout_ptn>(Trait::K, Trait::N));

        auto gm_tensor_as = asc::te::make_tensor(
            asc::te::make_mem_ptr(as),
            asc::te::make_frame_layout<asc::te::scalea_nd_layout_ptn>(Trait::M, scale_k));
        auto gm_tensor_bs = asc::te::make_tensor(
            asc::te::make_mem_ptr(bs),
            asc::te::make_frame_layout<asc::te::scaleb_dn_layout_ptn>(scale_k, Trait::N));
        auto gm_tensor_c = asc::te::make_tensor(
            asc::te::make_mem_ptr(c), asc::te::make_frame_layout<asc::te::nd_ext_layout_ptn>(Trait::M, Trait::N));

        // Slice single core tensor
        auto gm_single_tensor_a = gm_tensor_a.slice(
            asc::te::make_coord(m_iter_idx * Trait::single_core_m, 0),
            asc::te::make_shape(actual_single_core_m, Trait::single_core_k));
        auto gm_single_tensor_b = gm_tensor_b.slice(
            asc::te::make_coord(0, n_iter_idx * Trait::single_core_n),
            asc::te::make_shape(Trait::single_core_k, actual_single_core_n));
        auto gm_single_tensor_as = gm_tensor_as.slice(
            asc::te::make_coord(m_iter_idx * Trait::single_core_m, 0),
            asc::te::make_shape(actual_single_core_m, scale_k));
        auto gm_single_tensor_bs = gm_tensor_bs.slice(
            asc::te::make_coord(0, n_iter_idx * Trait::single_core_n),
            asc::te::make_shape(scale_k, actual_single_core_n));
        auto gm_single_tensor_c = gm_tensor_c.slice(
            asc::te::make_coord(m_iter_idx * Trait::single_core_m, n_iter_idx * Trait::single_core_n),
            asc::te::make_shape(actual_single_core_m, actual_single_core_n));

        process_loop(gm_single_tensor_a, gm_single_tensor_b, gm_single_tensor_as, gm_single_tensor_bs, gm_single_tensor_c);
    }

private:
    static constexpr uint32_t L1_DATA_INDEX = 0;  // A/B L1 ping/pong slots
    static constexpr uint32_t L1_SCALE_INDEX = 2; // As/Bs L1 ping/pong slots
    static constexpr uint32_t L0_INDEX = 4;      // L0 A/B ping/pong slots
    static constexpr uint32_t L0C_INDEX = 6;     // L0C single buffer slot

    __aicore__ inline uint32_t get_data_slot(uint32_t ping_pong_idx) const { return L1_DATA_INDEX + ping_pong_idx; }

    __aicore__ inline uint32_t get_scale_slot(uint32_t ping_pong_idx) const { return L1_SCALE_INDEX + ping_pong_idx; }

    __aicore__ inline uint32_t get_l0_slot(uint32_t ping_pong_idx) const { return L0_INDEX + ping_pong_idx; }

    // Main loop of matrix multiplication, including L1 prefetching, L1 to L0 copy, Mmad compute, and L0 to GM copy
    template <typename TensorA, typename TensorB, typename TensorAs, typename TensorBs, typename TensorC>
    __aicore__ inline void process_loop(
        const TensorA& a, const TensorB& b, const TensorAs& as, const TensorBs& bs, TensorC& c)
    {
        auto l1_layout_a = asc::te::make_frame_layout<asc::te::nz_layout_ptn, C0_ELEMENT_B4>(
            Trait::base_m, Trait::base_k * Trait::step_k);
        auto l1_layout_b = asc::te::make_frame_layout<asc::te::zn_layout_ptn, C0_ELEMENT_B4>(
            Trait::base_k * Trait::step_k, Trait::base_n);
        auto l1_layout_as = asc::te::make_frame_layout<asc::te::zz_layout_ptn, C0_ELEMENT_SCALE>(
            Trait::base_m, base_scale_k * Trait::step_k * Trait::scale_factor_k);
        auto l1_layout_bs = asc::te::make_frame_layout<asc::te::nn_layout_ptn, C0_ELEMENT_SCALE>(
            base_scale_k * Trait::step_k * Trait::scale_factor_k, Trait::base_n);

        // Malloc L1 and L0 buffers
        __cbuf__ fp4x2_e1m2_t l1_buf_a_ping[l1_size_a / 2];
        __cbuf__ fp4x2_e1m2_t l1_buf_a_pong[l1_size_a / 2];
        __cbuf__ fp4x2_e1m2_t l1_buf_b_ping[l1_size_b / 2];
        __cbuf__ fp4x2_e1m2_t l1_buf_b_pong[l1_size_b / 2];
        __cbuf__ fp8_e8m0_t l1_buf_as_ping[l1_size_as];
        __cbuf__ fp8_e8m0_t l1_buf_as_pong[l1_size_as];
        __cbuf__ fp8_e8m0_t l1_buf_bs_ping[l1_size_bs];
        __cbuf__ fp8_e8m0_t l1_buf_bs_pong[l1_size_bs];
        __ca__ fp4x2_e1m2_t l0_buf_a_ping[l0_size_a / 2];
        __ca__ fp4x2_e1m2_t l0_buf_a_pong[l0_size_a / 2];
        __cb__ fp4x2_e1m2_t l0_buf_b_ping[l0_size_b / 2];
        __cb__ fp4x2_e1m2_t l0_buf_b_pong[l0_size_b / 2];
        __cc__ float l0_buf_c[l0_size_c];

        auto l0_ptr_a_ping = asc::te::make_mem_ptr(l0_buf_a_ping);
        auto l0_ptr_a_pong = asc::te::make_mem_ptr(l0_buf_a_pong);
        auto l0_ptr_b_ping = asc::te::make_mem_ptr(l0_buf_b_ping);
        auto l0_ptr_b_pong = asc::te::make_mem_ptr(l0_buf_b_pong);
        auto l0_ptr_as_ping = asc::te::make_mem_ptr<asc::te::location::l0scalea, fp8_e8m0_t>(
            reinterpret_cast<uint64_t>(l0_buf_a_ping) / 16);
        auto l0_ptr_as_pong = asc::te::make_mem_ptr<asc::te::location::l0scalea, fp8_e8m0_t>(
            reinterpret_cast<uint64_t>(l0_buf_a_pong) / 16);
        auto l0_ptr_bs_ping = asc::te::make_mem_ptr<asc::te::location::l0scaleb, fp8_e8m0_t>(
            reinterpret_cast<uint64_t>(l0_buf_b_ping) / 16);
        auto l0_ptr_bs_pong = asc::te::make_mem_ptr<asc::te::location::l0scaleb, fp8_e8m0_t>(
            reinterpret_cast<uint64_t>(l0_buf_b_pong) / 16);

        auto l1_tensor_a_ping = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_a_ping), l1_layout_a);
        auto l1_tensor_a_pong = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_a_pong), l1_layout_a);
        auto l1_tensor_b_ping = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_b_ping), l1_layout_b);
        auto l1_tensor_b_pong = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_b_pong), l1_layout_b);
        auto l1_tensor_as_ping = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_as_ping), l1_layout_as);
        auto l1_tensor_as_pong = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_as_pong), l1_layout_as);
        auto l1_tensor_bs_ping = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_bs_ping), l1_layout_bs);
        auto l1_tensor_bs_pong = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_bs_pong), l1_layout_bs);

        // ============================================================
        // Mutex slot-lock synchronization: no preset needed (first asc_lock acquires a free slot).
        // ============================================================

        // ============================================================
        // N/M outer loops + K main loop
        // ============================================================
        for (uint32_t n_block_idx = 0; n_block_idx < n_loop_count; n_block_idx++) {
            uint16_t cur_n = (n_block_idx + 1 == n_loop_count) ? tail_n : Trait::base_n;
            for (uint32_t m_block_idx = 0; m_block_idx < m_loop_count; m_block_idx++) {
                uint16_t cur_m = (m_block_idx + 1 == m_loop_count) ? tail_m : Trait::base_m;
                // Reset the data_copy_in progress for each (m_block_idx, n_block_idx) tile.
                uint32_t data_next_k_chunk_idx = 0;  // K-direction chunk index for the next A/B data_copy_in.
                uint32_t scale_next_k_chunk_idx = 0; // K-direction chunk index for the next As/Bs data_copy_in.
                uint8_t data_copy_in_idx = 0;       // L1 buffer index written by the next A/B data_copy_in (0=Ping, 1=Pong).
                uint8_t scale_copy_in_idx = 0; // L1 buffer index written by the next As/Bs data_copy_in (0=Ping, 1=Pong).

                // Acquire L0C for Cube write (waits for Fixpipe to release it from the previous tile).
                asc_lock(PIPE_M, L0C_INDEX);

                // ---- Copy in the first chunk of A/B and As/Bs ----
                uint32_t data_write_slot = get_data_slot(data_copy_in_idx);
                asc_lock(PIPE_MTE2, data_write_slot); // A/B Ping writable.

                constexpr uint32_t step_cur_k = data_chunk_step * Trait::base_k;
                // A: GM -> L1
                asc::te::copy(
                    gm_to_l1_atom, l1_tensor_a_ping,
                    a.slice(
                        asc::te::make_coord(m_block_idx * Trait::base_m, data_next_k_chunk_idx * Trait::base_k),
                        asc::te::make_shape(cur_m, step_cur_k)));
                // B: GM -> L1
                asc::te::copy(
                    gm_to_l1_atom, l1_tensor_b_ping,
                    b.slice(
                        asc::te::make_coord(data_next_k_chunk_idx * Trait::base_k, n_block_idx * Trait::base_n),
                        asc::te::make_shape(step_cur_k, cur_n)));

                asc_unlock(PIPE_MTE2, data_write_slot); // A/B Ping written.
                data_next_k_chunk_idx += data_chunk_step;
                data_copy_in_idx ^= 1; // Write to Pong next time.

                uint32_t scale_write_slot = get_scale_slot(scale_copy_in_idx);
                asc_lock(PIPE_MTE2, scale_write_slot); // As/Bs Ping writable.

                constexpr uint32_t step_cur_scale_k =
                    align_even(AscendC::Std::ceil_div(scale_chunk_step * Trait::base_k, SCALE_CEIL_NUMBER));
                // scale_a: GM -> L1
                asc::te::copy(
                    gm_to_l1_atom, l1_tensor_as_ping,
                    as.slice(
                        asc::te::make_coord(m_block_idx * Trait::base_m, scale_next_k_chunk_idx * base_scale_k),
                        asc::te::make_shape(cur_m, step_cur_scale_k)));
                // scale_b: GM -> L1
                asc::te::copy(
                    gm_to_l1_atom, l1_tensor_bs_ping,
                    bs.slice(
                        asc::te::make_coord(scale_next_k_chunk_idx * base_scale_k, n_block_idx * Trait::base_n),
                        asc::te::make_shape(step_cur_scale_k, cur_n)));

                asc_unlock(PIPE_MTE2, scale_write_slot); // As/Bs Ping written.
                scale_next_k_chunk_idx += scale_chunk_step;
                scale_copy_in_idx ^= 1;

                auto l0_layout_c = asc::te::make_frame_layout<asc::te::nz_layout_ptn, C0_ELEMENT_L0C>(cur_m, cur_n);
                auto l0_tensor_c = asc::te::make_tensor(asc::te::make_mem_ptr(l0_buf_c), l0_layout_c);

                // ---- K-direction main loop ----
                for (uint32_t k_block_idx = 0; k_block_idx < k_loop_count; k_block_idx++) {
                    constexpr uint16_t cur_k = Trait::base_k;
                    // Determine the L1 read buffer (Ping/Pong) for the current k_block_idx.
                    uint32_t data_read_idx = (k_block_idx / data_chunk_step) % 2;
                    uint32_t scale_read_idx = (k_block_idx / scale_chunk_step) % 2;
                    uint32_t k_offset_in_data_chunk = k_block_idx % data_chunk_step;
                    uint32_t k_offset_in_scale_chunk = k_block_idx % scale_chunk_step;

                    const auto& l1_read_buf_a = (data_read_idx == 0) ? l1_tensor_a_ping : l1_tensor_a_pong;
                    const auto& l1_read_buf_b = (data_read_idx == 0) ? l1_tensor_b_ping : l1_tensor_b_pong;
                    const auto& l1_read_buf_as = (scale_read_idx == 0) ? l1_tensor_as_ping : l1_tensor_as_pong;
                    const auto& l1_read_buf_bs = (scale_read_idx == 0) ? l1_tensor_bs_ping : l1_tensor_bs_pong;

                    auto l0_layout_a = asc::te::make_frame_layout<asc::te::nz_layout_ptn, C0_ELEMENT_B4>(cur_m, cur_k);
                    auto l0_layout_b = asc::te::make_frame_layout<asc::te::zn_layout_ptn, C0_ELEMENT_B4>(cur_k, cur_n);
                    auto l0_tensor_a_ping = asc::te::make_tensor(l0_ptr_a_ping, l0_layout_a);
                    auto l0_tensor_a_pong = asc::te::make_tensor(l0_ptr_a_pong, l0_layout_a);
                    auto l0_tensor_b_ping = asc::te::make_tensor(l0_ptr_b_ping, l0_layout_b);
                    auto l0_tensor_b_pong = asc::te::make_tensor(l0_ptr_b_pong, l0_layout_b);

                    constexpr uint32_t cur_scale_k = align_even(AscendC::Std::ceil_div(cur_k, SCALE_CEIL_NUMBER));
                    auto l0_layout_as =
                        asc::te::make_frame_layout<asc::te::zz_layout_ptn, C0_ELEMENT_SCALE>(cur_m, cur_scale_k);
                    auto l0_layout_bs =
                        asc::te::make_frame_layout<asc::te::nn_layout_ptn, C0_ELEMENT_SCALE>(cur_scale_k, cur_n);
                    auto l0_tensor_as_ping = asc::te::make_tensor(l0_ptr_as_ping, l0_layout_as);
                    auto l0_tensor_as_pong = asc::te::make_tensor(l0_ptr_as_pong, l0_layout_as);
                    auto l0_tensor_bs_ping = asc::te::make_tensor(l0_ptr_bs_ping, l0_layout_bs);
                    auto l0_tensor_bs_pong = asc::te::make_tensor(l0_ptr_bs_pong, l0_layout_bs);

                    // Select the L0 double buffer.
                    const auto& l0_tensor_a = (mte1_db_flag == 0) ? l0_tensor_a_ping : l0_tensor_a_pong;
                    const auto& l0_tensor_b = (mte1_db_flag == 0) ? l0_tensor_b_ping : l0_tensor_b_pong;
                    const auto& l0_tensor_as = (mte1_db_flag == 0) ? l0_tensor_as_ping : l0_tensor_as_pong;
                    const auto& l0_tensor_bs = (mte1_db_flag == 0) ? l0_tensor_bs_ping : l0_tensor_bs_pong;

                    // ---- Reverse synchronization: wait for the previous Compute to release the L0 buffer ----
                    uint32_t l0_slot = get_l0_slot(mte1_db_flag);
                    asc_lock(PIPE_MTE1, l0_slot);

                    // ---- Forward synchronization ----
                    // Acquire the L1 chunk for reading at chunk start (waits for data_copy_in to finish writing it).
                    if (k_offset_in_data_chunk == 0) {
                        asc_lock(PIPE_MTE1, get_data_slot(data_read_idx));
                    }
                    if (k_offset_in_scale_chunk == 0) {
                        asc_lock(PIPE_MTE1, get_scale_slot(scale_read_idx));
                    }

                    // ---- data_load: L1 → L0 ----
                    // A:L1 -> L0A
                    asc::te::copy(
                        l1_to_l0a_atom, l0_tensor_a,
                        l1_read_buf_a.slice(
                            asc::te::make_coord(0, k_offset_in_data_chunk * Trait::base_k),
                            asc::te::make_shape(cur_m, cur_k)));
                    // B:L1 -> L0B
                    asc::te::copy(
                        l1_to_l0b_atom, l0_tensor_b,
                        l1_read_buf_b.slice(
                            asc::te::make_coord(k_offset_in_data_chunk * Trait::base_k, 0),
                            asc::te::make_shape(cur_k, cur_n)));

                    // scale_a: L1 -> L0AScale
                    asc::te::copy(
                        l1_to_l0scalea_atom, l0_tensor_as,
                        l1_read_buf_as.slice(
                            asc::te::make_coord(0, k_offset_in_scale_chunk * base_scale_k),
                            asc::te::make_shape(cur_m, cur_scale_k)));
                    // scale_b: L1 -> L0BScale
                    asc::te::copy(
                        l1_to_l0scaleb_atom, l0_tensor_bs,
                        l1_read_buf_bs.slice(
                            asc::te::make_coord(k_offset_in_scale_chunk * base_scale_k, 0),
                            asc::te::make_shape(cur_scale_k, cur_n)));

                    // ---- Reverse synchronization ----
                    // The current L1 chunk has been consumed; release it so data_copy_in can overwrite.
                    if (((k_offset_in_data_chunk + 1) == data_chunk_step) || (k_block_idx + 1 == k_loop_count)) {
                        asc_unlock(PIPE_MTE1, get_data_slot(data_read_idx));
                    }
                    if (((k_offset_in_scale_chunk + 1) == scale_chunk_step) || (k_block_idx + 1 == k_loop_count)) {
                        asc_unlock(PIPE_MTE1, get_scale_slot(scale_read_idx));
                    }

                    // L0 written, ready for Cube.
                    asc_unlock(PIPE_MTE1, l0_slot);

                    // ---- Compute: Mmad matrix multiply-accumulate ----
                    asc_lock(PIPE_M, l0_slot);
                    asc::te::mmad_params params{cur_m, cur_n, cur_k, asc::te::unit_flag_mode::disable, true};
                    params.init_with_zero = (k_block_idx == 0);

                    asc::te::mmad(mmad_atom.with(params), l0_tensor_c, l0_tensor_a, l0_tensor_b);
                    // M_MTE1 reverse synchronization: release the L0 buffer for the next data_load.
                    asc_unlock(PIPE_M, l0_slot);
                    mte1_db_flag ^= 1;

                    // ---- Copy in the next L1 chunk so data_copy_in overlaps with compute in the pipeline ----
                    // Trigger conditions:
                    //   (1) k_block_idx == 0: when computing the first K block, A1/B1 Pong has not been used and can be
                    //   copied in directly. (2) The last base_k of the current L1 chunk has been consumed, so the buffer
                    //   can be overwritten.
                    if (((k_block_idx == 0) || ((k_offset_in_data_chunk + 1) == data_chunk_step)) &&
                        data_next_k_chunk_idx < k_loop_count) {
                        const auto& l1_write_buf_a = (data_copy_in_idx == 0) ? l1_tensor_a_ping : l1_tensor_a_pong;
                        const auto& l1_write_buf_b = (data_copy_in_idx == 0) ? l1_tensor_b_ping : l1_tensor_b_pong;
                        uint32_t data_write_slot = get_data_slot(data_copy_in_idx);
                        asc_lock(PIPE_MTE2, data_write_slot);
                        // A: GM -> L1
                        asc::te::copy(
                            gm_to_l1_atom, l1_write_buf_a,
                            a.slice(
                                asc::te::make_coord(m_block_idx * Trait::base_m, data_next_k_chunk_idx * Trait::base_k),
                                asc::te::make_shape(cur_m, step_cur_k)));
                        // B: GM -> L1
                        asc::te::copy(
                            gm_to_l1_atom, l1_write_buf_b,
                            b.slice(
                                asc::te::make_coord(data_next_k_chunk_idx * Trait::base_k, n_block_idx * Trait::base_n),
                                asc::te::make_shape(step_cur_k, cur_n)));

                        asc_unlock(PIPE_MTE2, data_write_slot);
                        data_next_k_chunk_idx += data_chunk_step;
                        data_copy_in_idx ^= 1;
                    }
                    if (((k_block_idx == 0) || ((k_offset_in_scale_chunk + 1) == scale_chunk_step)) &&
                        scale_next_k_chunk_idx < k_loop_count) {
                        const auto& l1_write_buf_as = (scale_copy_in_idx == 0) ? l1_tensor_as_ping : l1_tensor_as_pong;
                        const auto& l1_write_buf_bs = (scale_copy_in_idx == 0) ? l1_tensor_bs_ping : l1_tensor_bs_pong;
                        uint32_t scale_write_slot = get_scale_slot(scale_copy_in_idx);
                        asc_lock(PIPE_MTE2, scale_write_slot);
                        // scale_a: GM -> L1
                        asc::te::copy(
                            gm_to_l1_atom, l1_write_buf_as,
                            as.slice(
                                asc::te::make_coord(m_block_idx * Trait::base_m, scale_next_k_chunk_idx * base_scale_k),
                                asc::te::make_shape(cur_m, step_cur_scale_k)));
                        // scale_b: GM -> L1
                        asc::te::copy(
                            gm_to_l1_atom, l1_write_buf_bs,
                            bs.slice(
                                asc::te::make_coord(scale_next_k_chunk_idx * base_scale_k, n_block_idx * Trait::base_n),
                                asc::te::make_shape(step_cur_scale_k, cur_n)));

                        asc_unlock(PIPE_MTE2, scale_write_slot);
                        scale_next_k_chunk_idx += scale_chunk_step;
                        scale_copy_in_idx ^= 1;
                    }
                }
                // ---- copy_out: L0C → GM ----
                asc_unlock(PIPE_M, L0C_INDEX);
                asc_lock(PIPE_FIX, L0C_INDEX);
                asc::te::l0c_to_gm_params fixpipe_params;
                // L0C -> GM
                asc::te::copy(
                    l0c_to_gm_atom.with(fixpipe_params),
                    c.slice(
                        asc::te::make_coord(m_block_idx * Trait::base_m, n_block_idx * Trait::base_n),
                        asc::te::make_shape(cur_m, cur_n)),
                    l0_tensor_c);

                asc_unlock(PIPE_FIX, L0C_INDEX);
            }
        }
    }

    __aicore__ inline void init_compute_params()
    {
        // ---- 1. Compute current-core M/N iteration indexes and GM start offset ----
        constexpr uint32_t m_iter = AscendC::Std::ceil_div(Trait::M, Trait::single_core_m);
        m_iter_idx = block_idx % m_iter;
        n_iter_idx = block_idx / m_iter;

        // ---- 2. Compute actual M/N dimensions of the current core ----
        // The last block may be smaller than single_core.
        actual_single_core_m = Trait::M - m_iter_idx * Trait::single_core_m;
        actual_single_core_m = actual_single_core_m < Trait::single_core_m ? actual_single_core_m : Trait::single_core_m;
        actual_single_core_n = Trait::N - n_iter_idx * Trait::single_core_n;
        actual_single_core_n = actual_single_core_n < Trait::single_core_n ? actual_single_core_n : Trait::single_core_n;

        // ---- 3. Compute the loop counts for the M/N/K dimensions ----
        m_loop_count = AscendC::Std::ceil_div(actual_single_core_m, Trait::base_m);
        n_loop_count = AscendC::Std::ceil_div(actual_single_core_n, Trait::base_n);
        k_loop_count = AscendC::Std::ceil_div(Trait::single_core_k, Trait::base_k);

        // ---- 4. Compute the M/N-direction tiling parameters ----
        tail_m = (actual_single_core_m % Trait::base_m != 0) ? actual_single_core_m % Trait::base_m : Trait::base_m;
        tail_n = (actual_single_core_n % Trait::base_n != 0) ? actual_single_core_n % Trait::base_n : Trait::base_n;
    }

private:
    uint32_t actual_single_core_m, actual_single_core_n;
    uint32_t m_iter_idx, n_iter_idx;
    uint32_t m_loop_count, n_loop_count, k_loop_count;
    uint32_t tail_m, tail_n;
    uint8_t mte1_db_flag = 0;

    static constexpr uint32_t scale_k = align_even(AscendC::Std::ceil_div(Trait::K, SCALE_CEIL_NUMBER));
    static constexpr uint32_t base_scale_k = align_even(AscendC::Std::ceil_div(Trait::base_k, SCALE_CEIL_NUMBER));

    static constexpr size_t l1_size_a = Trait::base_m * (Trait::base_k * Trait::step_k);
    static constexpr size_t l1_size_b = (Trait::base_k * Trait::step_k) * Trait::base_n;
    static constexpr size_t l1_size_as = Trait::base_m * (base_scale_k * Trait::step_k * Trait::scale_factor_k);
    static constexpr size_t l1_size_bs = (base_scale_k * Trait::step_k * Trait::scale_factor_k) * Trait::base_n;

    static constexpr size_t l0_size_a = Trait::base_m * Trait::base_k;
    static constexpr size_t l0_size_b = Trait::base_k * Trait::base_n;
    static constexpr size_t l0_size_c = Trait::base_m * Trait::base_n;

    static constexpr auto gm_to_l1_atom = asc::te::make_copy(asc::te::copy_gm_to_l1{});
    static constexpr auto l1_to_l0a_atom = asc::te::make_copy(asc::te::copy_l1_to_l0a{});
    static constexpr auto l1_to_l0b_atom = asc::te::make_copy(asc::te::copy_l1_to_l0b{});
    static constexpr auto l1_to_l0scalea_atom = asc::te::make_copy(asc::te::copy_l1_to_l0scalea{});
    static constexpr auto l1_to_l0scaleb_atom = asc::te::make_copy(asc::te::copy_l1_to_l0scaleb{});
    static constexpr auto l0c_to_gm_atom = asc::te::make_copy(asc::te::copy_l0c_to_gm{});
    static constexpr auto mmad_atom = asc::te::make_mmad(asc::te::mmad_operation{}, mmad_trait_mx{});

    static constexpr uint32_t data_chunk_step = Trait::step_k;
    static constexpr uint32_t scale_chunk_step = Trait::step_k * Trait::scale_factor_k;
};

template <typename Trait>
__global__ __cube__ void mmadmx_custom(
    __gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* as, __gm__ uint8_t* bs, __gm__ uint8_t* c)
{ // Use concrete data types.
    asc_init();
    kernel_mmadmx<Trait> op;
    op.process(
        reinterpret_cast<__gm__ fp4x2_e1m2_t*>(a), reinterpret_cast<__gm__ fp4x2_e1m2_t*>(b),
        reinterpret_cast<__gm__ fp8_e8m0_t*>(as), reinterpret_cast<__gm__ fp8_e8m0_t*>(bs),
        reinterpret_cast<__gm__ bfloat16_t*>(c));

    asc_sync_pipe(PIPE_ALL);
}

#endif


In [ ]:
%%writefile src/07_08_tensor_api_mxfp4_matmul/mmad_mx_high_performance.asc
/**
 * Copyright (c) 2026 Huawei Technologies Co., Ltd.
 * This program is free software, you can redistribute it and/or modify it under the terms and conditions of
 * CANN Open Software License Agreement Version 2.0 (the "License").
 * Please refer to the License for details. You may not use this file except in compliance with the License.
 * THIS FILE IS PROVIDED ON AN "AS IS" BASIS, WITHOUT WARRANTIES OF ANY KIND, EITHER EXPRESS OR IMPLIED,
 * INCLUDING BUT NOT LIMITED TO NON-INFRINGEMENT, MERCHANTABILITY, OR FITNESS FOR A PARTICULAR PURPOSE.
 * See LICENSE in the root of the software repository for the full text of the License.
 */

#include "mmad_mx_host.h"
#include "mmad_mx_high_performance_kernel.h"

namespace high_perf_config {
using Trait = kernel_trait<8192, 8192, 8192, 2048, 1024, 8192, 256, 256, 256, 2, 4>;
constexpr uint32_t NUM_BLOCKS = 32;

constexpr uint32_t scale_k_unaligned = (Trait::K + SCALE_CEIL_NUMBER - 1) / SCALE_CEIL_NUMBER;
constexpr uint32_t s_k = ((scale_k_unaligned + SCALE_ALIGN_NUMBER - 1) / SCALE_ALIGN_NUMBER) * SCALE_ALIGN_NUMBER;
}

int32_t main(int32_t argc, char *argv[])
{
    (void)argc;
    (void)argv;
    using namespace high_perf_config;

    constexpr size_t a_file_size = static_cast<size_t>(Trait::M * Trait::K) * sizeof(uint8_t) / 2;
    constexpr size_t b_file_size = static_cast<size_t>(Trait::N * Trait::K) * sizeof(uint8_t) / 2;
    constexpr size_t as_file_size = static_cast<size_t>(Trait::M * s_k) * sizeof(uint8_t);
    constexpr size_t bs_file_size = static_cast<size_t>(Trait::N * s_k) * sizeof(uint8_t);
    constexpr size_t c_file_size = Trait::M * Trait::N * sizeof(uint16_t);

    auto launch = [](uint8_t *a, uint8_t *b, uint8_t *as, uint8_t *bs, uint8_t *c, aclrtStream stream) {
        mmadmx_custom<Trait><<<NUM_BLOCKS, 0, stream>>>(a, b, as, bs, c);
    };
    return run_mmadmx_host(a_file_size, b_file_size, as_file_size, bs_file_size, c_file_size,
                         "./output/high_performance.bin", launch);
}


In [ ]:
!cd src/07_08_tensor_api_mxfp4_matmul && bash run.sh --case=high_performance

---
# 5. 优化手段详解

高性能 kernel 集成了 5 项关键优化手段。以下逐一讲解每项优化的原理、实现方式和具体效果。

## 5.1 多核切分

**原理**：Ascend 950PR 有 32 个 Cube Core。将大矩阵 `8192×8192` 的输出切分为 32 个子任务，每个核负责一个 `single_core_m × single_core_n = 2048×1024` 的输出区域，32 核并行计算。

**实现**：

```cpp
constexpr uint32_t m_iter = AscendC::Std::ceil_div(Trait::M, Trait::single_core_m);  // M方向4份
m_iter_idx = block_idx % m_iter;
n_iter_idx = block_idx / m_iter;                                  // N方向8份
```

每个核通过 GM Tensor 的 `slice` 取出自己负责的子矩阵，核间无数据依赖。支持尾块处理：最后一个核的 `actual_single_core_m`/`actual_single_core_n` 可能小于 `single_core_m`/`single_core_n`。

## 5.2 Mutex slot-lock 双缓冲流水

**原理**：单缓冲模式下，GM→L1 搬运、L1→L0 加载和 Cube 计算串行执行，Cube 大量时间在等数据。双缓冲（Ping-Pong）在 L1 和 L0 各分配两块缓冲区，当一块在被计算时，另一块可以同时搬运下一批数据，使 MTE2、MTE1、Cube 三级流水交叠执行。

**Mutex slot-lock 同步**：通过 `asc_lock/asc_unlock(PIPE_*, slot)` 管理缓冲区依赖。每个 slot 保护一块缓冲区，生产者 asc_lock 后写入、asc_unlock 释放，消费者 asc_lock 后读取、asc_unlock 释放。无需预置即可使用。

**缓冲区布局与 Slot 分配**：

| 层级 | 缓冲区 | Slot | 内容 |
| --- | --- | --- | --- |
| L1 | A/B Ping/Pong | 0/1 | `base_m × (base_k × step_k)` |
| L1 | ScaleA/ScaleB Ping/Pong | 2/3 | `base_m × (base_scale_k × step_k × scale_factor_k)` |
| L0 | A/B Ping/Pong | 4/5 | `base_m × base_k` / `base_k × base_n` |
| L0 | ScaleA/ScaleB | — | 与 L0A/B 共享物理内存 |
| L0C | C（单缓冲） | 6 | `base_m × base_n`（float） |

A/B Data 生命周期一致 → 绑定同一组 Slot（0/1）；ScaleA/ScaleB 生命周期一致但 chunk 更大 → 单独 Slot 组（2/3）。L0 的 A/B 和 Scale 共享物理内存，只需一组 Ping/Pong Slot（4/5）。

**四类同步事件**：

| 事件方向 | 含义 | Slot |
| --- | --- | --- |
| `MTE2→MTE1`（正向） | GM→L1 完成，通知 L1→L0 可读 | 0/1（Data）、2/3（Scale） |
| `MTE1→MTE2`（反向） | L1→L0 消费完，通知 GM→L1 可覆盖 | 同上 |
| `MTE1→M`（正向） | L1→L0 完成，通知 Cube 可算 | 4/5 |
| `M→MTE1`（反向） | Cube 消费完，通知 L1→L0 可写入 | 4/5 |

L0C 的同步独立使用 slot 6：Cube 写入前 `asc_lock(PIPE_M, 6)`，Fixpipe 搬出前 `asc_lock(PIPE_FIX, 6)`，搬出后 `asc_unlock(PIPE_FIX, 6)`。

**Mutex 同步流程图**：

```text
┌─────────┐ asc_lock(slot 0/1) ┌──────────┐ asc_lock(slot 4/5) ┌───────┐
│ GM→L1   │ ─────────────────→ │ L1→L0    │ ─────────────────→ │ Cube  │
│ MTE2    │ ←───────────────── │ MTE1     │ ←───────────────── │ M     │
│         │ asc_unlock(slot0/1)│          │ asc_unlock(slot4/5)│       │
└─────────┘                    └──────────┘                    └───────┘
     ↑                              ↑                               │
     │ asc_lock(slot 2/3)           │                               │ asc_lock(slot 6)
     │ for Scale                    │                               ↓
     └────────────────────────────────────────────────────── ┌──────────┐
                                                             │ L0C→GM   │
                                                             │ FIX      │
                                                             └──────────┘
```

**流水时序**：

```text
time     |--------------------------------------------------------------------------->
GM->L1   | A/B/As/Bs Ping | A/B/As/Bs Pong | A/B/As/Bs Ping | ...
L1->L0                     | L0 Ping load -| L0 Pong load -| ...
Cube                                      | Mmad Ping ----| Mmad Pong ----| ...
L0C->GM                                                    | Fixpipe ------|
```

**效果**：MTE2、MTE1、Cube 三级流水充分交叠。

## 5.3 大包搬运

**原理**：默认每次 GM→L1 只搬一个 `base_k` 块。大包搬运将多个 K block 合并为一次 Copy 操作，减少 GM→L1 的搬运指令数，降低 MTE2 的指令开销。

**两种大包粒度**：

- **step_k=2（Data 大包）**：A/B 一次搬入 2 个 base_k 块。`data_chunk_step = step_k = 2`，每 2 个 K block 才执行一次 GM→L1 搬运。L1 缓冲区大小为 `base_m × (base_k × step_k)`，后续 2 次 L1→L0 从中切出当前 base_k 块。
- **scale_factor_k=4（Scale 大包）**：Scale 一次覆盖 4 倍 A/B 的 K 范围。`scale_chunk_step = step_k × scale_factor_k = 8`，每 8 个 K block 才执行一次 Scale 的 GM→L1 搬运。

**为什么 Scale 需要更大的包**：Scale 数据量很小（K/32，每 32 个矩阵元素共享 1 个 scale），但搬运频率高（跟随每个 K block）。如果不加大包粒度，MTE2 会被大量小包 Scale 搬运指令占据。用更大的 `scale_factor_k` 可以减少 Scale 的搬运指令数，降低 MTE2 压力。

**效果**：GM→L1 搬运指令数从 `k_loop`（32 次/核）减少到 `k_loop/step_k`（16 次/核）for Data，`k_loop/scale_chunk_step`（4 次/核）for Scale。

## 5.4 块级同步

**原理**：双缓冲流水需要在"大包首元素等待数据就绪"和"大包末元素释放缓冲区"两个时机同步。如果在每个 K block 都做 asc_lock/asc_unlock，同步指令本身会成为开销。块级同步只在大包边界触发，减少同步指令数量。

**实现**：

```cpp
// 大包首元素：等待 GM→L1 写入完成
if (k_offset_in_data_chunk == 0) {
    asc_lock(PIPE_MTE1, get_data_slot(data_read_idx));
}
// 大包末元素：释放 L1 缓冲区，通知 GM→L1 可覆盖
if ((k_offset_in_data_chunk + 1 == data_chunk_step) || (k_block_idx + 1 == k_loop_count)) {
    asc_unlock(PIPE_MTE1, get_data_slot(data_read_idx));
}
```

Data 和 Scale 各自独立判断大包边界（`k_offset_in_data_chunk` 和 `k_offset_in_scale_chunk`），因为两者的 chunk 大小不同。

**效果**：同步指令数从 `k_loop × 2`（每 K block 两次）减少到 `k_loop/step_k × 2`（Data）+ `k_loop/scale_chunk_step × 2`（Scale），大幅降低 Scalar 开销。

## 5.5 F32→BF16 Fixpipe 转换写回

**原理**：Mmad 在 L0C 中以 `float` 精度累加，保证 K 方向累加不丢精度。输出需要 `bfloat16_t`，如果额外用 Vector 核做类型转换会增加流水线复杂度。Fixpipe 是 Cube 侧的随路转换功能，在 L0C→GM 的 `copy_l0c_to_gm` 中自动完成 float→bfloat16 的类型转换。

**实现**：

```cpp
l0c_to_gm_params fixpipe_params;
copy(l0c_to_gm_atom.with(fixpipe_params),
     c.slice(make_coord(m_block_idx * base_m, n_block_idx * base_n),
             make_shape(cur_m, cur_n)),
     l0_tensor_c);
```

L0C 使用单缓冲（slot 6），通过 `asc_lock(PIPE_M, 6)` 和 `asc_lock(PIPE_FIX, 6)` 实现 Cube 写入与 Fixpipe 搬出的互斥。


---
# 6. 性能分析

## 6.1 msOpProf 工具介绍

msOpProf 是单算子性能分析工具，包含 msopprof 和 msopprof simulator 两种使用方式。该工具协助用户定位算子内存、算子代码以及算子指令的异常，实现全方位的算子调优。当前支持基于不同运行模式（上板或仿真）和不同文件形式（可执行文件或算子二进制 .o 文件）进行性能数据的采集和自动解析。

基于可执行文件执行上板性能采集，可直接测定算子在昇腾 AI 处理器上的运行时间，适合在板环境中快速定位算子性能问题：

```bash
msopprof ./mmad_mx_high_performance
```

命令完成后，会在默认目录下生成以 `OPPROF_{timestamp}_XXX` 命名的文件夹，性能数据文件结构如下：

```text
├── dump                        # 原始的性能数据，用户无需关注
├── ArithmeticUtilization.csv   # cube/vector 指令 cycle 占比
├── L2Cache.csv                 # L2 Cache 命中率，影响 MTE2，建议合理规划数据搬运逻辑，增加命中率
├── Memory.csv                  # UB、L1 和主存储器读写带宽速率
├── MemoryL0.csv                # L0A、L0B 和 L0C 读写带宽速率
├── MemoryUB.csv                # Vector 和 Scalar 到 UB 的读写带宽速率
├── OpBasicInfo.csv             # 算子基础信息
├── PipeUtilization.csv         # 采集计算单元和搬运单元耗时和占比
├── ResourceConflictRatio.csv   # UB 上的 bank group、bank conflict 和资源冲突率在所有指令中的占比
└── visualize_data.bin          # MindStudio Insight 呈现文件
```

本节直接采用官方样例 `matmul_mxfp4_tensor_api_high_performance`（与本节 kernel 实现一致）在 Ascend 950PR 上通过 `msopprof` 采集的 `PipeUtilization.csv` 结果进行分析。


## 6.2 性能指标说明

`PipeUtilization.csv` 中各指标的含义如下：

| 指标 | 说明 |
| --- | --- |
| `Task Duration(μs)` | 整个任务执行的总时间，算子端到端执行时间以该参数为准 |
| `Block Num` | 使用的核数，也就是 kernel 启动的 block 数量 |
| `aicore_time(μs)` | AI Core 的平均执行时间 |
| `aic_mac_time(μs)` | Cube 计算单元执行时间，主要对应 `mmad` 阶段 |
| `aic_mac_ratio` | Cube 计算单元时间占比，反映计算单元利用率 |
| `aic_scalar_time(μs)` | Scalar 指令执行时间，反映循环调度、地址计算、参数配置等开销 |
| `aic_scalar_ratio` | Scalar 时间占比 |
| `aic_mte1_time(μs)` | MTE1 执行时间，主要对应 L1 到 L0A/L0B 的 `copy` |
| `aic_mte1_ratio` | MTE1 时间占比，反映 L1 到 L0 的数据搬运压力 |
| `aic_mte2_time(μs)` | MTE2 执行时间，主要对应 GM 到 L1 的 `copy` |
| `aic_mte2_ratio` | MTE2 时间占比，反映 GM 到 L1 的数据加载压力 |
| `aic_fixpipe_time(μs)` | L0C→GM 的 `copy` 执行时间，主要对应 L0C 到 GM 的结果写回 |
| `aic_fixpipe_ratio` | L0C→GM 的 `copy` 时间占比，反映结果写回的访存压力 |

## 6.3 样例实测结果

官方样例在 Ascend 950PR 上的采集结果：

| Case version | Task Duration(μs) | Block Num | aicore_time(μs) | aic_mac_time(μs) | aic_mac_ratio | aic_scalar_time(μs) | aic_scalar_ratio | aic_mte1_time(μs) | aic_mte1_ratio | aic_mte2_time(μs) | aic_mte2_ratio | aic_fixpipe_time(μs) | aic_fixpipe_ratio |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Tensor API MxMatmul | 682.316 | 32 | 681.54 | 640.591 | 0.94 | 63.466 | 0.093 | 315.894 | 0.464 | 589.308 | 0.865 | 31.646 | 0.046 |

可以看到，本样例的 Cube 计算时间占比（`aic_mac_ratio`）达到 **94.0%**，说明多核切分、大包搬运与双缓冲流水已让 Cube 计算单元接近满负荷工作。

## 6.4 Cube 理论性能计算

Ascend 950PR 主频 1.65GHz，MX-FP4 数据类型每 cycle 处理 `16×64×16` 次乘加运算。Cube 理论运算时间 $T_{cube}$ 为：

$$T_{cube} = \frac{M \times N \times K}{16 \times 64 \times 16 \times 1.65 \times 10^9 \times \text{核数}} = \frac{8192 \times 8192 \times 8192}{16384 \times 1.65 \times 10^9 \times 32} = 635.5 \, \mu s$$

实测 `aic_mac_time` 为 640.591 μs，相对理论值的误差 $E_{cube}$ 为：

$$E_{cube} = \frac{640.591 - 635.5}{635.5} = 0.80\%$$

Cube 实际计算时间与理论极限仅差 0.80%。

## 6.5 MTE2 带宽分析

**数据复用原理**：矩阵乘法中，输出矩阵 C 的每个元素 $C_{i,j}$ 需要 A 的第 i 行和 B 的第 j 列参与计算。分块计算时，同一输入数据块会被多个输出块复用：

- A 按 M 方向切成 `M/base_m = 32` 个行块，每个 A 行块参与 N 方向 32 个输出块的计算；
- B 按 N 方向切成 `N/base_n = 32` 个列块，每个 B 列块参与 M 方向 32 个输出块的计算；
- ScaleA/ScaleB 同理：K 方向每 32 个元素共享一个 scale（`scale_k = K/32 = 256`），ScaleA 形状 `[M, scale_k]`、ScaleB 形状 `[N, scale_k]`，各自参与 32 个输出块的计算。

由于 L1/L2Cache 容量有限，无法缓存所有输入数据，同一数据块会被多次从 HBM 搬运到 L2Cache/L1，导致数据重复搬运。

**读入数据总量**：A/B 为 `fp4x2_e1m2_t`（每 2 个 fp4 元素打包 1 byte，单元素 0.5B），ScaleA/ScaleB 为 `fp8_e8m0_t`（单元素 1B）。按 `M=N=K=8192`、`scale_k=256`、`base_m=base_n=256` 计算：

$$D_{total} = \frac{N}{base_n} \times M \times K \times 0.5B + \frac{M}{base_m} \times K \times N \times 0.5B + \frac{N}{base_n} \times M \times scale_k \times 1B + \frac{M}{base_m} \times scale_k \times N \times 1B$$

$$= 1GB + 1GB + 64MB + 64MB = 2.125GB$$

**MTE2 理论耗时**：Ascend 950PR 的 L2Cache 峰值带宽约 5TB/s，HBM（对应 GM）带宽约 1.6TB/s。最理想情况下首次访问从 HBM 获取数据并缓存到 L2Cache，后续访问直接从 L2Cache 读取：

- 首次从 HBM 读入的数据总量：$D_{HBM} = M \times K \times 0.5B + K \times N \times 0.5B + M \times scale_k \times 1B + N \times scale_k \times 1B = 32MB + 32MB + 2MB + 2MB = 68MB$
- 从 L2Cache 读入的数据总量：$D_{L2Cache} = D_{total} - D_{HBM} = 2.125GB - 68MB \approx 2.057GB$

$$T_{MTE2} = \frac{D_{HBM}}{1.6TB/s} + \frac{D_{L2Cache}}{5TB/s} = \frac{68MB}{1.6TB/s} + \frac{2.057GB}{5TB/s} \approx 42.5\mu s + 411.4\mu s = 453.9\mu s$$

实测 `aic_mte2_time` 为 589.308 μs，误差 $E_{MTE2}$ 为：

$$E_{MTE2} = \frac{589.308 - 453.9}{453.9} = 29.8\%$$

> **单位说明**：带宽单位采用十进制，1 TB/s = 10^12 B/s。

## 6.6 L2Cache 优化讨论

Ascend 950PR 的 L2Cache 大小为 128MB，无法缓存所有输入数据，部分数据在搬运时会发生 L2Cache miss，需要从 HBM 获取，导致实际 MTE2 耗时（589.308 μs）高于理论值（453.9 μs）。优化方向：调整 `single_core_m`/`single_core_n` 等切分参数，使数据块更友好地利用 L2Cache，提高 L2Cache 命中率。此为开放性讨论，本节不要求实现。

---
# 7. 课后练习与课后实践

## 课后练习

1. MxFP4 中 K 方向每 ___ 个元素共享一个 scale，scale_k 需对齐到 ___ Byte。
2. Mutex slot-lock 同步中，`asc_lock` 和 `asc_unlock` 分别由哪个流水线调用？本例为何需要 7 个 slot？
3. `scale_factor_k=4` 的作用是什么？为什么 Scale 数据需要比 A/B 更大的搬运包？
4. `init_with_zero=true` 在什么场景下设置？
5. L0ScaleA 与 L0A 的物理内存关系是什么？

> 答案见 `answer/07_08_tensor_api_mxfp4_matmul/answer.md`，可通过下方 code cell 查看。

In [ ]:
!cat ../answer/07_08_tensor_api_mxfp4_matmul/answer.md

## 课后实践：随路量化

在 Stage 2 的高性能 MxFP4 kernel 基础上，添加 per-channel 量化功能：
- **当前输出**：C `[M, N]` `bfloat16_t`（L0C float → Fixpipe → BF16）
- **目标输出**：C `[M, N]` `int8`（L0C float → Fixpipe + 量化 → INT8）
- **新增输入**：`quant_scale [1, N]`、`quant_offset [1, N]`，类型为 `uint64_t`（Fixpipe 量化参数打包格式，scale/offset 按位打包，详见 `gen_data_quant.py` 中的打包注释）

### 关键 API 差异

```cpp
// 当前（无量化，3 参数 copy）：
copy(l0c_to_gm_atom.with(fixpipe_params), gm_c_slice, l0_c_tensor);

// 目标（带量化，4 参数 copy，附加量化参数张量）：
copy(l0c_to_gm_atom.with(fixpipe_params), gm_c_slice, l0_c_tensor, l1_quant_tensor);
```

### TODO 模板说明

实践题模板 `mmad_mx_quant_practice_kernel.h` 中有 4 处 TODO：

1. 函数签名扩展：添加 `quant_scale`/`quant_offset` 参数
2. L1 缓冲区分配：为量化参数分配 L1 空间
3. GM→L1 搬运量化参数：copy 量化参数到 L1
4. L0C→GM 量化 copy 调用：填写四参数 copy

写入实践题工程文件后，补全 TODO 并运行验证：

In [ ]:
%%writefile src/07_08_tensor_api_mxfp4_matmul/mmad_mx_quant_practice_kernel.h
#ifndef MMAD_MX_QUANT_PRACTICE_KERNEL_H
#define MMAD_MX_QUANT_PRACTICE_KERNEL_H

#include "c_api/asc_simd.h"
#include "tensor_api/tensor.h"
#include "utils/std/cmath.h"

__aicore__ __inline__ constexpr uint32_t align_even(uint32_t a) { return (a + 1) / 2 * 2; }

constexpr uint32_t SCALE_CEIL_NUMBER = 32;
constexpr uint32_t SCALE_ALIGN_NUMBER = 2;
constexpr uint32_t C0_ELEMENT_SCALE = 2;
constexpr uint32_t C0_ELEMENT_L0C = 16;
constexpr uint32_t C0_ELEMENT_B4 = 64;
constexpr uint32_t CUBE_BLOCK = 16;

template <
    uint32_t M_, uint32_t N_, uint32_t K_, uint32_t single_core_m_, uint32_t single_core_n_, uint32_t single_core_k_,
    uint32_t base_m_, uint32_t base_n_, uint32_t base_k_, uint32_t step_k_, uint32_t scale_factor_k_>
struct kernel_trait {
    static constexpr uint32_t M = M_;
    static constexpr uint32_t N = N_;
    static constexpr uint32_t K = K_;

    static constexpr uint32_t single_core_m = single_core_m_;
    static constexpr uint32_t single_core_n = single_core_n_;
    static constexpr uint32_t single_core_k = single_core_k_;

    static constexpr uint32_t base_m = base_m_;
    static constexpr uint32_t base_n = base_n_;
    static constexpr uint32_t base_k = base_k_;

    static constexpr uint32_t step_k = step_k_;
    static constexpr uint32_t scale_factor_k = scale_factor_k_;
};

constexpr asc::te::mmad_trait MX_MMAD_TRAIT =
    asc::te::mmad_trait{0, false, false, true, asc::te::mmad_type::mx};
struct mmad_trait_mx {
    using trait_type = asc::te::mmad_trait;
    static constexpr const trait_type value = MX_MMAD_TRAIT;
};

template <typename Trait>
class kernel_mmadmx_quant {
public:
    __aicore__ inline kernel_mmadmx_quant() {}

    // TODO(1): 在 process 函数签名中添加量化参数 quant_scale 和 quant_offset
    // 提示：添加 __gm__ uint64_t* quant_scale, __gm__ uint64_t* quant_offset
    //       输出类型已从 __gm__ bfloat16_t* c 改为 __gm__ int8_t* c
    __aicore__ inline void process(
        __gm__ fp4x2_e1m2_t* a, __gm__ fp4x2_e1m2_t* b, __gm__ fp8_e8m0_t* as, __gm__ fp8_e8m0_t* bs,
        __gm__ int8_t* c
        /* TODO(1): 在此添加 __gm__ uint64_t* quant_scale, __gm__ uint64_t* quant_offset */)
    {
        init_compute_params();

        // Init GM tensor
        auto gm_tensor_a = asc::te::make_tensor(
            asc::te::make_mem_ptr(a), asc::te::make_frame_layout<asc::te::nd_ext_layout_ptn>(Trait::M, Trait::K));
        auto gm_tensor_b = asc::te::make_tensor(
            asc::te::make_mem_ptr(b), asc::te::make_frame_layout<asc::te::dn_ext_layout_ptn>(Trait::K, Trait::N));

        auto gm_tensor_as = asc::te::make_tensor(
            asc::te::make_mem_ptr(as),
            asc::te::make_frame_layout<asc::te::scalea_nd_layout_ptn>(Trait::M, scale_k));
        auto gm_tensor_bs = asc::te::make_tensor(
            asc::te::make_mem_ptr(bs),
            asc::te::make_frame_layout<asc::te::scaleb_dn_layout_ptn>(scale_k, Trait::N));
        auto gm_tensor_c = asc::te::make_tensor(
            asc::te::make_mem_ptr(c), asc::te::make_frame_layout<asc::te::nd_ext_layout_ptn>(Trait::M, Trait::N));

        // Init GM quant tensors (scale and offset, shape [1, N])
        auto gm_quant_scale = asc::te::make_tensor(
            asc::te::make_mem_ptr(quant_scale),
            asc::te::make_frame_layout<asc::te::nd_layout_ptn>(1, Trait::N));
        auto gm_quant_offset = asc::te::make_tensor(
            asc::te::make_mem_ptr(quant_offset),
            asc::te::make_frame_layout<asc::te::nd_layout_ptn>(1, Trait::N));

        // Slice single core tensor
        auto gm_single_tensor_a = gm_tensor_a.slice(
            asc::te::make_coord(m_iter_idx * Trait::single_core_m, 0),
            asc::te::make_shape(actual_single_core_m, Trait::single_core_k));
        auto gm_single_tensor_b = gm_tensor_b.slice(
            asc::te::make_coord(0, n_iter_idx * Trait::single_core_n),
            asc::te::make_shape(Trait::single_core_k, actual_single_core_n));
        auto gm_single_tensor_as = gm_tensor_as.slice(
            asc::te::make_coord(m_iter_idx * Trait::single_core_m, 0),
            asc::te::make_shape(actual_single_core_m, scale_k));
        auto gm_single_tensor_bs = gm_tensor_bs.slice(
            asc::te::make_coord(0, n_iter_idx * Trait::single_core_n),
            asc::te::make_shape(scale_k, actual_single_core_n));
        auto gm_single_tensor_c = gm_tensor_c.slice(
            asc::te::make_coord(m_iter_idx * Trait::single_core_m, n_iter_idx * Trait::single_core_n),
            asc::te::make_shape(actual_single_core_m, actual_single_core_n));

        process_loop(gm_single_tensor_a, gm_single_tensor_b, gm_single_tensor_as, gm_single_tensor_bs, gm_single_tensor_c,
                    gm_quant_scale, gm_quant_offset);
    }

private:
    static constexpr uint32_t L1_DATA_INDEX = 0;   // A/B L1 ping/pong slots
    static constexpr uint32_t L1_SCALE_INDEX = 2;  // As/Bs L1 ping/pong slots
    static constexpr uint32_t L0_INDEX = 4;       // L0 A/B ping/pong slots
    static constexpr uint32_t L0C_INDEX = 6;      // L0C single buffer slot
    static constexpr uint32_t QUANT_INDEX = 7;    // Quant L1 buffer slot

    __aicore__ inline uint32_t get_data_slot(uint32_t ping_pong_idx) const { return L1_DATA_INDEX + ping_pong_idx; }

    __aicore__ inline uint32_t get_scale_slot(uint32_t ping_pong_idx) const { return L1_SCALE_INDEX + ping_pong_idx; }

    __aicore__ inline uint32_t get_l0_slot(uint32_t ping_pong_idx) const { return L0_INDEX + ping_pong_idx; }

    // Main loop of matrix multiplication, including L1 prefetching, L1 to L0 copy, Mmad compute, and L0 to GM copy
    template <typename TensorA, typename TensorB, typename TensorAs, typename TensorBs, typename TensorC,
              typename TensorQuantScale, typename TensorQuantOffset>
    __aicore__ inline void process_loop(
        const TensorA& a, const TensorB& b, const TensorAs& as, const TensorBs& bs, TensorC& c,
        const TensorQuantScale& quant_scale, const TensorQuantOffset& quant_offset)
    {
        auto l1_layout_a = asc::te::make_frame_layout<asc::te::nz_layout_ptn, C0_ELEMENT_B4>(
            Trait::base_m, Trait::base_k * Trait::step_k);
        auto l1_layout_b = asc::te::make_frame_layout<asc::te::zn_layout_ptn, C0_ELEMENT_B4>(
            Trait::base_k * Trait::step_k, Trait::base_n);
        auto l1_layout_as = asc::te::make_frame_layout<asc::te::zz_layout_ptn, C0_ELEMENT_SCALE>(
            Trait::base_m, base_scale_k * Trait::step_k * Trait::scale_factor_k);
        auto l1_layout_bs = asc::te::make_frame_layout<asc::te::nn_layout_ptn, C0_ELEMENT_SCALE>(
            base_scale_k * Trait::step_k * Trait::scale_factor_k, Trait::base_n);

        // Malloc L1 and L0 buffers
        __cbuf__ fp4x2_e1m2_t l1_buf_a_ping[l1_size_a / 2];
        __cbuf__ fp4x2_e1m2_t l1_buf_a_pong[l1_size_a / 2];
        __cbuf__ fp4x2_e1m2_t l1_buf_b_ping[l1_size_b / 2];
        __cbuf__ fp4x2_e1m2_t l1_buf_b_pong[l1_size_b / 2];
        __cbuf__ fp8_e8m0_t l1_buf_as_ping[l1_size_as];
        __cbuf__ fp8_e8m0_t l1_buf_as_pong[l1_size_as];
        __cbuf__ fp8_e8m0_t l1_buf_bs_ping[l1_size_bs];
        __cbuf__ fp8_e8m0_t l1_buf_bs_pong[l1_size_bs];
        __ca__ fp4x2_e1m2_t l0_buf_a_ping[l0_size_a / 2];
        __ca__ fp4x2_e1m2_t l0_buf_a_pong[l0_size_a / 2];
        __cb__ fp4x2_e1m2_t l0_buf_b_ping[l0_size_b / 2];
        __cb__ fp4x2_e1m2_t l0_buf_b_pong[l0_size_b / 2];
        __cc__ float l0_buf_c[l0_size_c];

        // TODO(2): 添加量化参数的 L1 缓冲区
        // 提示：声明两个 __cbuf__ uint64_t 数组，大小均为 Trait::base_n
        //   __cbuf__ uint64_t l1_buf_quant_scale[Trait::base_n];
        //   __cbuf__ uint64_t l1_buf_quant_offset[Trait::base_n];
        /* TODO(2): 在此声明 l1_buf_quant_scale 和 l1_buf_quant_offset */

        auto l0_ptr_a_ping = asc::te::make_mem_ptr(l0_buf_a_ping);
        auto l0_ptr_a_pong = asc::te::make_mem_ptr(l0_buf_a_pong);
        auto l0_ptr_b_ping = asc::te::make_mem_ptr(l0_buf_b_ping);
        auto l0_ptr_b_pong = asc::te::make_mem_ptr(l0_buf_b_pong);
        auto l0_ptr_as_ping = asc::te::make_mem_ptr<asc::te::location::l0scalea, fp8_e8m0_t>(
            reinterpret_cast<uint64_t>(l0_buf_a_ping) / 16);
        auto l0_ptr_as_pong = asc::te::make_mem_ptr<asc::te::location::l0scalea, fp8_e8m0_t>(
            reinterpret_cast<uint64_t>(l0_buf_a_pong) / 16);
        auto l0_ptr_bs_ping = asc::te::make_mem_ptr<asc::te::location::l0scaleb, fp8_e8m0_t>(
            reinterpret_cast<uint64_t>(l0_buf_b_ping) / 16);
        auto l0_ptr_bs_pong = asc::te::make_mem_ptr<asc::te::location::l0scaleb, fp8_e8m0_t>(
            reinterpret_cast<uint64_t>(l0_buf_b_pong) / 16);

        auto l1_tensor_a_ping = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_a_ping), l1_layout_a);
        auto l1_tensor_a_pong = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_a_pong), l1_layout_a);
        auto l1_tensor_b_ping = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_b_ping), l1_layout_b);
        auto l1_tensor_b_pong = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_b_pong), l1_layout_b);
        auto l1_tensor_as_ping = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_as_ping), l1_layout_as);
        auto l1_tensor_as_pong = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_as_pong), l1_layout_as);
        auto l1_tensor_bs_ping = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_bs_ping), l1_layout_bs);
        auto l1_tensor_bs_pong = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_bs_pong), l1_layout_bs);

        // L1 quant tensors (依赖 TODO(2) 中声明的缓冲区)
        auto l1_quant_scale = asc::te::make_tensor(
            asc::te::make_mem_ptr(l1_buf_quant_scale),
            asc::te::make_frame_layout<asc::te::nd_layout_ptn, uint64_t>(1, Trait::base_n));
        auto l1_quant_offset = asc::te::make_tensor(
            asc::te::make_mem_ptr(l1_buf_quant_offset),
            asc::te::make_frame_layout<asc::te::nd_layout_ptn, uint64_t>(1, Trait::base_n));

        // ============================================================
        // Mutex slot-lock synchronization: no preset needed (first asc_lock acquires a free slot).
        // ============================================================

        // ============================================================
        // N/M outer loops + K main loop
        // ============================================================
        for (uint32_t n_block_idx = 0; n_block_idx < n_loop_count; n_block_idx++) {
            uint16_t cur_n = (n_block_idx + 1 == n_loop_count) ? tail_n : Trait::base_n;
            for (uint32_t m_block_idx = 0; m_block_idx < m_loop_count; m_block_idx++) {
                uint16_t cur_m = (m_block_idx + 1 == m_loop_count) ? tail_m : Trait::base_m;
                // Reset the data_copy_in progress for each (m_block_idx, n_block_idx) tile.
                uint32_t data_next_k_chunk_idx = 0;  // K-direction chunk index for the next A/B data_copy_in.
                uint32_t scale_next_k_chunk_idx = 0; // K-direction chunk index for the next As/Bs data_copy_in.
                uint8_t data_copy_in_idx = 0;       // L1 buffer index written by the next A/B data_copy_in (0=Ping, 1=Pong).
                uint8_t scale_copy_in_idx = 0; // L1 buffer index written by the next As/Bs data_copy_in (0=Ping, 1=Pong).

                // Acquire L0C for Cube write (waits for Fixpipe to release it from the previous tile).
                asc_lock(PIPE_M, L0C_INDEX);

                // ---- Copy in the first chunk of A/B and As/Bs ----
                uint32_t data_write_slot = get_data_slot(data_copy_in_idx);
                asc_lock(PIPE_MTE2, data_write_slot); // A/B Ping writable.

                constexpr uint32_t step_cur_k = data_chunk_step * Trait::base_k;
                // A: GM -> L1
                asc::te::copy(
                    gm_to_l1_atom, l1_tensor_a_ping,
                    a.slice(
                        asc::te::make_coord(m_block_idx * Trait::base_m, data_next_k_chunk_idx * Trait::base_k),
                        asc::te::make_shape(cur_m, step_cur_k)));
                // B: GM -> L1
                asc::te::copy(
                    gm_to_l1_atom, l1_tensor_b_ping,
                    b.slice(
                        asc::te::make_coord(data_next_k_chunk_idx * Trait::base_k, n_block_idx * Trait::base_n),
                        asc::te::make_shape(step_cur_k, cur_n)));

                asc_unlock(PIPE_MTE2, data_write_slot); // A/B Ping written.
                data_next_k_chunk_idx += data_chunk_step;
                data_copy_in_idx ^= 1; // Write to Pong next time.

                uint32_t scale_write_slot = get_scale_slot(scale_copy_in_idx);
                asc_lock(PIPE_MTE2, scale_write_slot); // As/Bs Ping writable.

                constexpr uint32_t step_cur_scale_k =
                    align_even(AscendC::Std::ceil_div(scale_chunk_step * Trait::base_k, SCALE_CEIL_NUMBER));
                // scale_a: GM -> L1
                asc::te::copy(
                    gm_to_l1_atom, l1_tensor_as_ping,
                    as.slice(
                        asc::te::make_coord(m_block_idx * Trait::base_m, scale_next_k_chunk_idx * base_scale_k),
                        asc::te::make_shape(cur_m, step_cur_scale_k)));
                // scale_b: GM -> L1
                asc::te::copy(
                    gm_to_l1_atom, l1_tensor_bs_ping,
                    bs.slice(
                        asc::te::make_coord(scale_next_k_chunk_idx * base_scale_k, n_block_idx * Trait::base_n),
                        asc::te::make_shape(step_cur_scale_k, cur_n)));

                asc_unlock(PIPE_MTE2, scale_write_slot); // As/Bs Ping written.
                scale_next_k_chunk_idx += scale_chunk_step;
                scale_copy_in_idx ^= 1;

                auto l0_layout_c = asc::te::make_frame_layout<asc::te::nz_layout_ptn, C0_ELEMENT_L0C>(cur_m, cur_n);
                auto l0_tensor_c = asc::te::make_tensor(asc::te::make_mem_ptr(l0_buf_c), l0_layout_c);

                // ---- K-direction main loop ----
                for (uint32_t k_block_idx = 0; k_block_idx < k_loop_count; k_block_idx++) {
                    constexpr uint16_t cur_k = Trait::base_k;
                    // Determine the L1 read buffer (Ping/Pong) for the current k_block_idx.
                    uint32_t data_read_idx = (k_block_idx / data_chunk_step) % 2;
                    uint32_t scale_read_idx = (k_block_idx / scale_chunk_step) % 2;
                    uint32_t k_offset_in_data_chunk = k_block_idx % data_chunk_step;
                    uint32_t k_offset_in_scale_chunk = k_block_idx % scale_chunk_step;

                    const auto& l1_read_buf_a = (data_read_idx == 0) ? l1_tensor_a_ping : l1_tensor_a_pong;
                    const auto& l1_read_buf_b = (data_read_idx == 0) ? l1_tensor_b_ping : l1_tensor_b_pong;
                    const auto& l1_read_buf_as = (scale_read_idx == 0) ? l1_tensor_as_ping : l1_tensor_as_pong;
                    const auto& l1_read_buf_bs = (scale_read_idx == 0) ? l1_tensor_bs_ping : l1_tensor_bs_pong;

                    auto l0_layout_a = asc::te::make_frame_layout<asc::te::nz_layout_ptn, C0_ELEMENT_B4>(cur_m, cur_k);
                    auto l0_layout_b = asc::te::make_frame_layout<asc::te::zn_layout_ptn, C0_ELEMENT_B4>(cur_k, cur_n);
                    auto l0_tensor_a_ping = asc::te::make_tensor(l0_ptr_a_ping, l0_layout_a);
                    auto l0_tensor_a_pong = asc::te::make_tensor(l0_ptr_a_pong, l0_layout_a);
                    auto l0_tensor_b_ping = asc::te::make_tensor(l0_ptr_b_ping, l0_layout_b);
                    auto l0_tensor_b_pong = asc::te::make_tensor(l0_ptr_b_pong, l0_layout_b);

                    constexpr uint32_t cur_scale_k = align_even(AscendC::Std::ceil_div(cur_k, SCALE_CEIL_NUMBER));
                    auto l0_layout_as =
                        asc::te::make_frame_layout<asc::te::zz_layout_ptn, C0_ELEMENT_SCALE>(cur_m, cur_scale_k);
                    auto l0_layout_bs =
                        asc::te::make_frame_layout<asc::te::nn_layout_ptn, C0_ELEMENT_SCALE>(cur_scale_k, cur_n);
                    auto l0_tensor_as_ping = asc::te::make_tensor(l0_ptr_as_ping, l0_layout_as);
                    auto l0_tensor_as_pong = asc::te::make_tensor(l0_ptr_as_pong, l0_layout_as);
                    auto l0_tensor_bs_ping = asc::te::make_tensor(l0_ptr_bs_ping, l0_layout_bs);
                    auto l0_tensor_bs_pong = asc::te::make_tensor(l0_ptr_bs_pong, l0_layout_bs);

                    // Select the L0 double buffer.
                    const auto& l0_tensor_a = (mte1_db_flag == 0) ? l0_tensor_a_ping : l0_tensor_a_pong;
                    const auto& l0_tensor_b = (mte1_db_flag == 0) ? l0_tensor_b_ping : l0_tensor_b_pong;
                    const auto& l0_tensor_as = (mte1_db_flag == 0) ? l0_tensor_as_ping : l0_tensor_as_pong;
                    const auto& l0_tensor_bs = (mte1_db_flag == 0) ? l0_tensor_bs_ping : l0_tensor_bs_pong;

                    // ---- Reverse synchronization: wait for the previous Compute to release the L0 buffer ----
                    uint32_t l0_slot = get_l0_slot(mte1_db_flag);
                    asc_lock(PIPE_MTE1, l0_slot);

                    // ---- Forward synchronization ----
                    // Acquire the L1 chunk for reading at chunk start (waits for data_copy_in to finish writing it).
                    if (k_offset_in_data_chunk == 0) {
                        asc_lock(PIPE_MTE1, get_data_slot(data_read_idx));
                    }
                    if (k_offset_in_scale_chunk == 0) {
                        asc_lock(PIPE_MTE1, get_scale_slot(scale_read_idx));
                    }

                    // ---- data_load: L1 -> L0 ----
                    // A:L1 -> L0A
                    asc::te::copy(
                        l1_to_l0a_atom, l0_tensor_a,
                        l1_read_buf_a.slice(
                            asc::te::make_coord(0, k_offset_in_data_chunk * Trait::base_k),
                            asc::te::make_shape(cur_m, cur_k)));
                    // B:L1 -> L0B
                    asc::te::copy(
                        l1_to_l0b_atom, l0_tensor_b,
                        l1_read_buf_b.slice(
                            asc::te::make_coord(k_offset_in_data_chunk * Trait::base_k, 0),
                            asc::te::make_shape(cur_k, cur_n)));

                    // scale_a: L1 -> L0AScale
                    asc::te::copy(
                        l1_to_l0scalea_atom, l0_tensor_as,
                        l1_read_buf_as.slice(
                            asc::te::make_coord(0, k_offset_in_scale_chunk * base_scale_k),
                            asc::te::make_shape(cur_m, cur_scale_k)));
                    // scale_b: L1 -> L0BScale
                    asc::te::copy(
                        l1_to_l0scaleb_atom, l0_tensor_bs,
                        l1_read_buf_bs.slice(
                            asc::te::make_coord(k_offset_in_scale_chunk * base_scale_k, 0),
                            asc::te::make_shape(cur_scale_k, cur_n)));

                    // ---- Reverse synchronization ----
                    // The current L1 chunk has been consumed; release it so data_copy_in can overwrite.
                    if (((k_offset_in_data_chunk + 1) == data_chunk_step) || (k_block_idx + 1 == k_loop_count)) {
                        asc_unlock(PIPE_MTE1, get_data_slot(data_read_idx));
                    }
                    if (((k_offset_in_scale_chunk + 1) == scale_chunk_step) || (k_block_idx + 1 == k_loop_count)) {
                        asc_unlock(PIPE_MTE1, get_scale_slot(scale_read_idx));
                    }

                    // L0 written, ready for Cube.
                    asc_unlock(PIPE_MTE1, l0_slot);

                    // ---- Compute: Mmad matrix multiply-accumulate ----
                    asc_lock(PIPE_M, l0_slot);
                    asc::te::mmad_params params{cur_m, cur_n, cur_k, asc::te::unit_flag_mode::disable, true};
                    params.init_with_zero = (k_block_idx == 0);

                    asc::te::mmad(mmad_atom.with(params), l0_tensor_c, l0_tensor_a, l0_tensor_b);
                    // M_MTE1 reverse synchronization: release the L0 buffer for the next data_load.
                    asc_unlock(PIPE_M, l0_slot);
                    mte1_db_flag ^= 1;

                    // ---- Copy in the next L1 chunk so data_copy_in overlaps with compute in the pipeline ----
                    // Trigger conditions:
                    //   (1) k_block_idx == 0: when computing the first K block, A1/B1 Pong has not been used and can be
                    //   copied in directly. (2) The last base_k of the current L1 chunk has been consumed, so the buffer
                    //   can be overwritten.
                    if (((k_block_idx == 0) || ((k_offset_in_data_chunk + 1) == data_chunk_step)) &&
                        data_next_k_chunk_idx < k_loop_count) {
                        const auto& l1_write_buf_a = (data_copy_in_idx == 0) ? l1_tensor_a_ping : l1_tensor_a_pong;
                        const auto& l1_write_buf_b = (data_copy_in_idx == 0) ? l1_tensor_b_ping : l1_tensor_b_pong;
                        uint32_t data_write_slot = get_data_slot(data_copy_in_idx);
                        asc_lock(PIPE_MTE2, data_write_slot);
                        // A: GM -> L1
                        asc::te::copy(
                            gm_to_l1_atom, l1_write_buf_a,
                            a.slice(
                                asc::te::make_coord(m_block_idx * Trait::base_m, data_next_k_chunk_idx * Trait::base_k),
                                asc::te::make_shape(cur_m, step_cur_k)));
                        // B: GM -> L1
                        asc::te::copy(
                            gm_to_l1_atom, l1_write_buf_b,
                            b.slice(
                                asc::te::make_coord(data_next_k_chunk_idx * Trait::base_k, n_block_idx * Trait::base_n),
                                asc::te::make_shape(step_cur_k, cur_n)));

                        asc_unlock(PIPE_MTE2, data_write_slot);
                        data_next_k_chunk_idx += data_chunk_step;
                        data_copy_in_idx ^= 1;
                    }
                    if (((k_block_idx == 0) || ((k_offset_in_scale_chunk + 1) == scale_chunk_step)) &&
                        scale_next_k_chunk_idx < k_loop_count) {
                        const auto& l1_write_buf_as = (scale_copy_in_idx == 0) ? l1_tensor_as_ping : l1_tensor_as_pong;
                        const auto& l1_write_buf_bs = (scale_copy_in_idx == 0) ? l1_tensor_bs_ping : l1_tensor_bs_pong;
                        uint32_t scale_write_slot = get_scale_slot(scale_copy_in_idx);
                        asc_lock(PIPE_MTE2, scale_write_slot);
                        // scale_a: GM -> L1
                        asc::te::copy(
                            gm_to_l1_atom, l1_write_buf_as,
                            as.slice(
                                asc::te::make_coord(m_block_idx * Trait::base_m, scale_next_k_chunk_idx * base_scale_k),
                                asc::te::make_shape(cur_m, step_cur_scale_k)));
                        // scale_b: GM -> L1
                        asc::te::copy(
                            gm_to_l1_atom, l1_write_buf_bs,
                            bs.slice(
                                asc::te::make_coord(scale_next_k_chunk_idx * base_scale_k, n_block_idx * Trait::base_n),
                                asc::te::make_shape(step_cur_scale_k, cur_n)));

                        asc_unlock(PIPE_MTE2, scale_write_slot);
                        scale_next_k_chunk_idx += scale_chunk_step;
                        scale_copy_in_idx ^= 1;
                    }
                }
                // ---- copy quant params: GM -> L1 ----
                asc_lock(PIPE_MTE2, QUANT_INDEX);
                // TODO(3): 将 quant_scale 从 GM 拷贝到 L1
                // 提示：使用 gm_to_l1_atom 将 quant_scale 的当前 N 块拷贝到 l1_quant_scale
                //   copy(gm_to_l1_atom, l1_quant_scale,
                //     quant_scale.slice(make_coord(0, n_block_idx * Trait::base_n), make_shape(1, cur_n)))
                /* TODO(3): 在此填写 copy 语句 */
                asc_unlock(PIPE_MTE2, QUANT_INDEX);

                // ---- copy_out: L0C -> GM (with quantization) ----
                asc_unlock(PIPE_M, L0C_INDEX);
                asc_lock(PIPE_FIX, QUANT_INDEX);
                asc_lock(PIPE_FIX, L0C_INDEX);
                asc::te::l0c_to_gm_params fixpipe_params;
                // TODO(4): 将 L0C -> GM 的 3 参数 copy 改为 4 参数 copy（附加量化张量）
                // 提示：在第 3 个参数 l0_tensor_c 之后添加第 4 个参数 l1_quant_scale
                //   copy(l0c_to_gm_atom.with(fixpipe_params), gm_c_slice, l0_tensor_c, l1_quant_scale)
                asc::te::copy(
                    l0c_to_gm_atom.with(fixpipe_params),
                    c.slice(
                        asc::te::make_coord(m_block_idx * Trait::base_m, n_block_idx * Trait::base_n),
                        asc::te::make_shape(cur_m, cur_n)),
                    l0_tensor_c
                    /* TODO(4): 在此添加第 4 个参数 l1_quant_scale */);
                asc_unlock(PIPE_FIX, L0C_INDEX);
                asc_unlock(PIPE_FIX, QUANT_INDEX);
            }
        }
    }

    __aicore__ inline void init_compute_params()
    {
        // ---- 1. Compute current-core M/N iteration indexes and GM start offset ----
        constexpr uint32_t m_iter = AscendC::Std::ceil_div(Trait::M, Trait::single_core_m);
        m_iter_idx = block_idx % m_iter;
        n_iter_idx = block_idx / m_iter;

        // ---- 2. Compute actual M/N dimensions of the current core ----
        // The last block may be smaller than single_core.
        actual_single_core_m = Trait::M - m_iter_idx * Trait::single_core_m;
        actual_single_core_m = actual_single_core_m < Trait::single_core_m ? actual_single_core_m : Trait::single_core_m;
        actual_single_core_n = Trait::N - n_iter_idx * Trait::single_core_n;
        actual_single_core_n = actual_single_core_n < Trait::single_core_n ? actual_single_core_n : Trait::single_core_n;

        // ---- 3. Compute the loop counts for the M/N/K dimensions ----
        m_loop_count = AscendC::Std::ceil_div(actual_single_core_m, Trait::base_m);
        n_loop_count = AscendC::Std::ceil_div(actual_single_core_n, Trait::base_n);
        k_loop_count = AscendC::Std::ceil_div(Trait::single_core_k, Trait::base_k);

        // ---- 4. Compute the M/N-direction tiling parameters ----
        tail_m = (actual_single_core_m % Trait::base_m != 0) ? actual_single_core_m % Trait::base_m : Trait::base_m;
        tail_n = (actual_single_core_n % Trait::base_n != 0) ? actual_single_core_n % Trait::base_n : Trait::base_n;
    }

private:
    uint32_t actual_single_core_m, actual_single_core_n;
    uint32_t m_iter_idx, n_iter_idx;
    uint32_t m_loop_count, n_loop_count, k_loop_count;
    uint32_t tail_m, tail_n;
    uint8_t mte1_db_flag = 0;

    static constexpr uint32_t scale_k = align_even(AscendC::Std::ceil_div(Trait::K, SCALE_CEIL_NUMBER));
    static constexpr uint32_t base_scale_k = align_even(AscendC::Std::ceil_div(Trait::base_k, SCALE_CEIL_NUMBER));

    static constexpr size_t l1_size_a = Trait::base_m * (Trait::base_k * Trait::step_k);
    static constexpr size_t l1_size_b = (Trait::base_k * Trait::step_k) * Trait::base_n;
    static constexpr size_t l1_size_as = Trait::base_m * (base_scale_k * Trait::step_k * Trait::scale_factor_k);
    static constexpr size_t l1_size_bs = (base_scale_k * Trait::step_k * Trait::scale_factor_k) * Trait::base_n;

    static constexpr size_t l0_size_a = Trait::base_m * Trait::base_k;
    static constexpr size_t l0_size_b = Trait::base_k * Trait::base_n;
    static constexpr size_t l0_size_c = Trait::base_m * Trait::base_n;

    static constexpr auto gm_to_l1_atom = asc::te::make_copy(asc::te::copy_gm_to_l1{});
    static constexpr auto l1_to_l0a_atom = asc::te::make_copy(asc::te::copy_l1_to_l0a{});
    static constexpr auto l1_to_l0b_atom = asc::te::make_copy(asc::te::copy_l1_to_l0b{});
    static constexpr auto l1_to_l0scalea_atom = asc::te::make_copy(asc::te::copy_l1_to_l0scalea{});
    static constexpr auto l1_to_l0scaleb_atom = asc::te::make_copy(asc::te::copy_l1_to_l0scaleb{});
    static constexpr auto l0c_to_gm_atom = asc::te::make_copy(asc::te::copy_l0c_to_gm{});
    static constexpr auto mmad_atom = asc::te::make_mmad(asc::te::mmad_operation{}, mmad_trait_mx{});

    static constexpr uint32_t data_chunk_step = Trait::step_k;
    static constexpr uint32_t scale_chunk_step = Trait::step_k * Trait::scale_factor_k;
};

template <typename Trait>
__global__ __cube__ void mmadmx_quant_custom(
    __gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* as, __gm__ uint8_t* bs, __gm__ uint8_t* c,
    __gm__ uint8_t* quant_scale, __gm__ uint8_t* quant_offset)
{
    asc_init();
    kernel_mmadmx_quant<Trait> op;
    op.process(
        reinterpret_cast<__gm__ fp4x2_e1m2_t*>(a), reinterpret_cast<__gm__ fp4x2_e1m2_t*>(b),
        reinterpret_cast<__gm__ fp8_e8m0_t*>(as), reinterpret_cast<__gm__ fp8_e8m0_t*>(bs),
        reinterpret_cast<__gm__ int8_t*>(c), reinterpret_cast<__gm__ uint64_t*>(quant_scale),
        reinterpret_cast<__gm__ uint64_t*>(quant_offset));

    asc_sync_pipe(PIPE_ALL);
}

#endif


In [ ]:
%%writefile src/07_08_tensor_api_mxfp4_matmul/mmad_mx_quant_practice.asc
/**
 * Copyright (c) 2026 Huawei Technologies Co., Ltd.
 * This program is free software, you can redistribute it and/or modify it under the terms and conditions of
 * CANN Open Software License Agreement Version 2.0 (the "License").
 * Please refer to the License for details. You may not use this file except in compliance with the License.
 * THIS FILE IS PROVIDED ON AN "AS IS" BASIS, WITHOUT WARRANTIES OF ANY KIND, EITHER EXPRESS OR IMPLIED,
 * INCLUDING BUT NOT LIMITED TO NON-INFRINGEMENT, MERCHANTABILITY, OR FITNESS FOR A PARTICULAR PURPOSE.
 * See LICENSE in the root of the software repository for the full text of the License.
 */

#include "mmad_mx_host.h"
#include "mmad_mx_quant_practice_kernel.h"

namespace quant_practice_config {
using Trait = kernel_trait<8192, 8192, 8192, 2048, 1024, 8192, 256, 256, 256, 2, 4>;
constexpr uint32_t NUM_BLOCKS = 32;

constexpr uint32_t scale_k_unaligned = (Trait::K + SCALE_CEIL_NUMBER - 1) / SCALE_CEIL_NUMBER;
constexpr uint32_t s_k = ((scale_k_unaligned + SCALE_ALIGN_NUMBER - 1) / SCALE_ALIGN_NUMBER) * SCALE_ALIGN_NUMBER;
}

int32_t main(int32_t argc, char *argv[])
{
    (void)argc;
    (void)argv;
    using namespace quant_practice_config;

    // 文件大小：输入与 high_performance 相同，输出改为 INT8
    constexpr size_t a_file_size = static_cast<size_t>(Trait::M * Trait::K) * sizeof(uint8_t) / 2;
    constexpr size_t b_file_size = static_cast<size_t>(Trait::N * Trait::K) * sizeof(uint8_t) / 2;
    constexpr size_t as_file_size = static_cast<size_t>(Trait::M * s_k) * sizeof(uint8_t);
    constexpr size_t bs_file_size = static_cast<size_t>(Trait::N * s_k) * sizeof(uint8_t);
    // TODO(Host-1): 输出文件大小从 BF16 (sizeof(uint16_t)) 改为 INT8 (sizeof(int8_t))
    // 提示：c_file_size = Trait::M * Trait::N * sizeof(int8_t)
    constexpr size_t c_file_size = Trait::M * Trait::N * sizeof(uint16_t); /* TODO(Host-1): 改为 sizeof(int8_t) */

    // TODO(Host-2): 定义量化参数的文件大小
    // 提示：quant_scale 和 quant_offset 的 shape 均为 [1, N]，类型为 uint64_t
    //   constexpr size_t quant_file_size = Trait::N * sizeof(uint64_t);
    /* TODO(Host-2): 在此定义 quant_file_size */

    CHECK_ACL_RET(aclInit(nullptr));
    int32_t device_id = 0;
    CHECK_ACL_RET(aclrtSetDevice(device_id));

    aclrtStream stream = nullptr;
    CHECK_ACL_RET(aclrtCreateStream(&stream));

    uint8_t *a_host = nullptr, *b_host = nullptr, *as_host = nullptr, *bs_host = nullptr, *c_host = nullptr;
    uint8_t *a_device = nullptr, *b_device = nullptr, *as_device = nullptr, *bs_device = nullptr, *c_device = nullptr;

    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&a_host), a_file_size));
    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&b_host), b_file_size));
    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&as_host), as_file_size));
    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&bs_host), bs_file_size));
    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&c_host), c_file_size));

    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&a_device), a_file_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&b_device), b_file_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&as_device), as_file_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&bs_device), bs_file_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&c_device), c_file_size, ACL_MEM_MALLOC_HUGE_FIRST));

    // TODO(Host-3): 为量化参数分配 host 和 device 内存
    // 提示：quant_scale_host, quant_offset_host, quant_scale_device, quant_offset_device
    //   aclrtMallocHost / aclrtMalloc，大小为 quant_file_size
    /* TODO(Host-3): 在此分配量化参数内存 */

    size_t file_size = a_file_size;
    if (!ReadFile("./input/x1_gm.bin", file_size, a_host, a_file_size)) { return 1; }
    file_size = b_file_size;
    if (!ReadFile("./input/x2_gm.bin", file_size, b_host, b_file_size)) { return 1; }
    file_size = as_file_size;
    if (!ReadFile("./input/x1_scale_gm.bin", file_size, as_host, as_file_size)) { return 1; }
    file_size = bs_file_size;
    if (!ReadFile("./input/x2_scale_gm.bin", file_size, bs_host, bs_file_size)) { return 1; }

    // TODO(Host-4): 读取量化参数文件
    // 提示：ReadFile("./input/quant_scale.bin", quant_file_size, quant_scale_host, quant_file_size)
    //       ReadFile("./input/quant_offset.bin", quant_file_size, quant_offset_host, quant_file_size)
    /* TODO(Host-4): 在此读取 quant_scale.bin 和 quant_offset.bin */

    CHECK_ACL_RET(aclrtMemcpy(a_device, a_file_size, a_host, a_file_size, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL_RET(aclrtMemcpy(b_device, b_file_size, b_host, b_file_size, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL_RET(aclrtMemcpy(as_device, as_file_size, as_host, as_file_size, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL_RET(aclrtMemcpy(bs_device, bs_file_size, bs_host, bs_file_size, ACL_MEMCPY_HOST_TO_DEVICE));

    // TODO(Host-5): 将量化参数从 host 拷贝到 device
    // 提示：aclrtMemcpy(quant_scale_device, quant_file_size, quant_scale_host, quant_file_size, ACL_MEMCPY_HOST_TO_DEVICE)
    /* TODO(Host-5): 在此拷贝量化参数到 device */

    // TODO(Host-6): 在 launch 中将量化参数传递给 kernel
    // 提示：mmadmx_quant_custom<Trait><<<NUM_BLOCKS, 0, stream>>>(
    //   a, b, as, bs, c, quant_scale, quant_offset)
    auto launch = [](uint8_t *a, uint8_t *b, uint8_t *as, uint8_t *bs, uint8_t *c, aclrtStream stream) {
        mmadmx_quant_custom<Trait><<<NUM_BLOCKS, 0, stream>>>(a, b, as, bs, c);
    };
    launch(a_device, b_device, as_device, bs_device, c_device, stream);
    CHECK_ACL_RET(aclrtSynchronizeStream(stream));

    CHECK_ACL_RET(aclrtMemcpy(c_host, c_file_size, c_device, c_file_size, ACL_MEMCPY_DEVICE_TO_HOST));
    if (!WriteFile("./output/quant_practice.bin", c_host, c_file_size)) { return 1; }

    (void)aclrtFree(a_device); (void)aclrtFree(b_device);
    (void)aclrtFree(as_device); (void)aclrtFree(bs_device);
    (void)aclrtFree(c_device);
    (void)aclrtFreeHost(a_host); (void)aclrtFreeHost(b_host);
    (void)aclrtFreeHost(as_host); (void)aclrtFreeHost(bs_host);
    (void)aclrtFreeHost(c_host);

    // TODO(Host-7): 释放量化参数的 host 和 device 内存
    /* TODO(Host-7): 在此释放量化参数内存 */

    (void)aclrtDestroyStream(stream);
    (void)aclrtResetDevice(device_id);
    (void)aclFinalize();
    return 0;
}


In [ ]:
%%writefile src/07_08_tensor_api_mxfp4_matmul/scripts/gen_data_quant.py
import os

import numpy as np
import ml_dtypes
import en_dtypes

bfloat16 = ml_dtypes.bfloat16
fp4_e1m2x2 = en_dtypes.float4_e1m2


def pack_two_fp4(scale_matrix):
    scale_matrix_row = scale_matrix.shape[0]
    scale_matrix_col = scale_matrix.shape[1]
    scale_matrix_bin = scale_matrix.flatten()
    scale_matrix_high = scale_matrix_bin[::2].view(np.uint8)
    scale_matrix_low = scale_matrix_bin[1::2].view(np.uint8)
    low_bits = (scale_matrix_low & 0x0F) << 4
    high_bits = scale_matrix_high & 0x0F
    combined = low_bits | high_bits
    scale_matrix_bin = combined.reshape(scale_matrix_row, scale_matrix_col // 2)
    return scale_matrix_bin


def main():
    m, n, k = 8192, 8192, 8192
    sk = (int)(np.ceil(k / 64) * 2)

    os.makedirs("input", exist_ok=True)
    os.makedirs("output", exist_ok=True)

    np.random.seed(42)
    x1_gm = np.random.uniform(-1, 2, [m, k]).astype(fp4_e1m2x2)
    x2_gm = np.random.uniform(-1, 2, [k, n]).astype(fp4_e1m2x2)

    x1_scale_gm = np.random.randint(127, 130, [m, sk]).astype(np.uint8)
    x2_scale_gm = np.random.randint(127, 130, [sk, n]).astype(np.uint8)

    x1_mx = 2 ** (x1_scale_gm.astype(np.float64) - 127)
    x2_mx = 2 ** (x2_scale_gm.astype(np.float64) - 127)
    x1_full = np.zeros([m, k], dtype=np.float64)
    x2_full = np.zeros([k, n], dtype=np.float64)

    for i in range(x1_gm.shape[1]):
        x1_full[:, i] = x1_gm[:, i] * x1_mx[:, i // 32]
        x2_full[i, :] = x2_gm[i, :] * x2_mx[i // 32, :]

    golden_f64 = np.matmul(x1_full.astype(np.float64), x2_full.astype(np.float64))

    # 量化参数：per-channel 量化 (F32 -> INT8)
    quant_scale = np.ones([1, n], dtype=np.float32) * 0.01
    quant_offset = np.zeros([1, n], dtype=np.float32)

    # Fixpipe 量化参数为 uint64 打包格式（参考 SetQuantVector 接口的参数格式）：
    #   quant = (scale & 0xFFFFE000) | (1 << 46) | ((offset & 0x1FF) << 37)
    # scale 按 float32 位截取高 19 位（1 符号 + 8 指数 + 10 尾数），bit46 为标志位；
    # offset 四舍五入取整并截断到 [-256, 255]，按 9 位存于 bit37~45。
    scale_bits = np.frombuffer(quant_scale, np.uint32).reshape(1, n) & np.uint32(0xFFFFE000)
    scale_hw = scale_bits.view(np.float32)  # 硬件实际生效的截断后 scale
    offset_int = np.clip(np.round(quant_offset), -256, 255).astype(np.int64)
    quant_scale_tensor = scale_bits.astype(np.uint64) | (np.uint64(1) << np.uint64(46))
    quant_scale_tensor |= (offset_int.astype(np.uint64) & np.uint64(0x1FF)) << np.uint64(37)
    quant_offset_tensor = quant_scale_tensor.copy()  # offset 已打包进 quant 参数，kernel 当前仅使用 scale

    # INT8 golden = clip(round(golden_f64 * scale + offset), -128, 127)
    # golden 使用截断后的 scale 与取整后的 offset，保证与硬件 Fixpipe 量化语义一致
    golden_quant = np.clip(
        np.round(golden_f64 * scale_hw + offset_int), -128, 127
    ).astype(np.int8)

    x2_gm = x2_gm.transpose()
    x2_scale_gm = x2_scale_gm.transpose()
    x1_gm_packed = pack_two_fp4(x1_gm)
    x2_gm_packed = pack_two_fp4(x2_gm)
    x1_gm_packed.tofile("input/x1_gm.bin")
    x2_gm_packed.tofile("input/x2_gm.bin")
    x1_scale_gm.tofile("input/x1_scale_gm.bin")
    x2_scale_gm.tofile("input/x2_scale_gm.bin")
    quant_scale_tensor.tofile("input/quant_scale.bin")
    quant_offset_tensor.tofile("input/quant_offset.bin")
    golden_quant.tofile("output/golden_quant.bin")

    print(f"generated MxFP4 input data: A[{m},{k}], B[{k},{n}], ScaleA[{m},{sk}], ScaleB[{sk},{n}]")
    print(f"generated quant params (uint64 packed): Scale[1,{n}], Offset[1,{n}]")
    print(f"golden INT8 output saved to output/golden_quant.bin")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/07_08_tensor_api_mxfp4_matmul/scripts/verify_result_quant.py
import sys

import numpy as np

# 量化误差容忍：元素间允许 ±1 的舍入差异
ERROR_TOL = 0.01  # 1% 的元素允许相差超过 1


def main():
    if len(sys.argv) != 2:
        raise SystemExit("Usage: python3 verify_result_quant.py output/<case>.bin")

    output_path = sys.argv[1]
    golden_path = "output/golden_quant.bin"

    output = np.fromfile(output_path, dtype=np.int8).reshape(-1)
    golden = np.fromfile(golden_path, dtype=np.int8).reshape(-1)

    if output.size != golden.size:
        raise SystemExit(f"size mismatch: output {output.size}, golden {golden.size}")

    diff = np.abs(output.astype(np.int32) - golden.astype(np.int32))
    # 误差超过 1 的元素视为错误（量化舍入允许 ±1）
    error_mask = diff > 1
    error_count = np.count_nonzero(error_mask)
    total = golden.size

    print(f"Total elements compared: {total}")
    print(f"Total error elements (diff > 1): {error_count}")

    if error_count > 0:
        error_idx = np.where(error_mask)[0]
        for idx in error_idx[:100]:
            golden_val = int(golden[idx])
            output_val = int(output[idx])
            print(f"data index: {idx:06d}, expected: {golden_val}, actual: {output_val}, "
                  f"diff: {abs(output_val - golden_val)}")

    error_ratio = float(error_count) / total
    print(f"error ratio: {error_ratio:.4f}, tolerance: {ERROR_TOL:.4f}")

    if error_ratio > ERROR_TOL:
        raise SystemExit("verify failed!")
    print("test pass!")


if __name__ == "__main__":
    main()


### 运行实践

补全 TODO 后运行：

```bash
cd src/07_08_tensor_api_mxfp4_matmul && bash run.sh --case=quant_practice
```

也可以直接运行参考实现验证：

In [ ]:
%%writefile src/07_08_tensor_api_mxfp4_matmul/mmad_mx_quant_answer_kernel.h
#ifndef MMAD_MX_QUANT_ANSWER_KERNEL_H
#define MMAD_MX_QUANT_ANSWER_KERNEL_H

#include "c_api/asc_simd.h"
#include "tensor_api/tensor.h"
#include "utils/std/cmath.h"

__aicore__ __inline__ constexpr uint32_t align_even(uint32_t a) { return (a + 1) / 2 * 2; }

constexpr uint32_t SCALE_CEIL_NUMBER = 32;
constexpr uint32_t SCALE_ALIGN_NUMBER = 2;
constexpr uint32_t C0_ELEMENT_SCALE = 2;
constexpr uint32_t C0_ELEMENT_L0C = 16;
constexpr uint32_t C0_ELEMENT_B4 = 64;
constexpr uint32_t CUBE_BLOCK = 16;

template <
    uint32_t M_, uint32_t N_, uint32_t K_, uint32_t single_core_m_, uint32_t single_core_n_, uint32_t single_core_k_,
    uint32_t base_m_, uint32_t base_n_, uint32_t base_k_, uint32_t step_k_, uint32_t scale_factor_k_>
struct kernel_trait {
    static constexpr uint32_t M = M_;
    static constexpr uint32_t N = N_;
    static constexpr uint32_t K = K_;

    static constexpr uint32_t single_core_m = single_core_m_;
    static constexpr uint32_t single_core_n = single_core_n_;
    static constexpr uint32_t single_core_k = single_core_k_;

    static constexpr uint32_t base_m = base_m_;
    static constexpr uint32_t base_n = base_n_;
    static constexpr uint32_t base_k = base_k_;

    static constexpr uint32_t step_k = step_k_;
    static constexpr uint32_t scale_factor_k = scale_factor_k_;
};

constexpr asc::te::mmad_trait MX_MMAD_TRAIT =
    asc::te::mmad_trait{0, false, false, true, asc::te::mmad_type::mx};
struct mmad_trait_mx {
    using trait_type = asc::te::mmad_trait;
    static constexpr const trait_type value = MX_MMAD_TRAIT;
};

template <typename Trait>
class kernel_mmadmx_quant {
public:
    __aicore__ inline kernel_mmadmx_quant() {}

    // [Answer-1] process 签名添加量化参数 quant_scale 和 quant_offset，输出类型改为 int8_t
    __aicore__ inline void process(
        __gm__ fp4x2_e1m2_t* a, __gm__ fp4x2_e1m2_t* b, __gm__ fp8_e8m0_t* as, __gm__ fp8_e8m0_t* bs,
        __gm__ int8_t* c, __gm__ uint64_t* quant_scale, __gm__ uint64_t* quant_offset)
    {
        init_compute_params();

        // Init GM tensor
        auto gm_tensor_a = asc::te::make_tensor(
            asc::te::make_mem_ptr(a), asc::te::make_frame_layout<asc::te::nd_ext_layout_ptn>(Trait::M, Trait::K));
        auto gm_tensor_b = asc::te::make_tensor(
            asc::te::make_mem_ptr(b), asc::te::make_frame_layout<asc::te::dn_ext_layout_ptn>(Trait::K, Trait::N));

        auto gm_tensor_as = asc::te::make_tensor(
            asc::te::make_mem_ptr(as),
            asc::te::make_frame_layout<asc::te::scalea_nd_layout_ptn>(Trait::M, scale_k));
        auto gm_tensor_bs = asc::te::make_tensor(
            asc::te::make_mem_ptr(bs),
            asc::te::make_frame_layout<asc::te::scaleb_dn_layout_ptn>(scale_k, Trait::N));
        auto gm_tensor_c = asc::te::make_tensor(
            asc::te::make_mem_ptr(c), asc::te::make_frame_layout<asc::te::nd_ext_layout_ptn>(Trait::M, Trait::N));

        // Init GM quant tensors (scale and offset, shape [1, N])
        auto gm_quant_scale = asc::te::make_tensor(
            asc::te::make_mem_ptr(quant_scale),
            asc::te::make_frame_layout<asc::te::nd_layout_ptn>(1, Trait::N));
        auto gm_quant_offset = asc::te::make_tensor(
            asc::te::make_mem_ptr(quant_offset),
            asc::te::make_frame_layout<asc::te::nd_layout_ptn>(1, Trait::N));

        // Slice single core tensor
        auto gm_single_tensor_a = gm_tensor_a.slice(
            asc::te::make_coord(m_iter_idx * Trait::single_core_m, 0),
            asc::te::make_shape(actual_single_core_m, Trait::single_core_k));
        auto gm_single_tensor_b = gm_tensor_b.slice(
            asc::te::make_coord(0, n_iter_idx * Trait::single_core_n),
            asc::te::make_shape(Trait::single_core_k, actual_single_core_n));
        auto gm_single_tensor_as = gm_tensor_as.slice(
            asc::te::make_coord(m_iter_idx * Trait::single_core_m, 0),
            asc::te::make_shape(actual_single_core_m, scale_k));
        auto gm_single_tensor_bs = gm_tensor_bs.slice(
            asc::te::make_coord(0, n_iter_idx * Trait::single_core_n),
            asc::te::make_shape(scale_k, actual_single_core_n));
        auto gm_single_tensor_c = gm_tensor_c.slice(
            asc::te::make_coord(m_iter_idx * Trait::single_core_m, n_iter_idx * Trait::single_core_n),
            asc::te::make_shape(actual_single_core_m, actual_single_core_n));

        process_loop(gm_single_tensor_a, gm_single_tensor_b, gm_single_tensor_as, gm_single_tensor_bs, gm_single_tensor_c,
                    gm_quant_scale, gm_quant_offset);
    }

private:
    static constexpr uint32_t L1_DATA_INDEX = 0;   // A/B L1 ping/pong slots
    static constexpr uint32_t L1_SCALE_INDEX = 2;  // As/Bs L1 ping/pong slots
    static constexpr uint32_t L0_INDEX = 4;       // L0 A/B ping/pong slots
    static constexpr uint32_t L0C_INDEX = 6;      // L0C single buffer slot
    static constexpr uint32_t QUANT_INDEX = 7;    // Quant L1 buffer slot

    __aicore__ inline uint32_t get_data_slot(uint32_t ping_pong_idx) const { return L1_DATA_INDEX + ping_pong_idx; }

    __aicore__ inline uint32_t get_scale_slot(uint32_t ping_pong_idx) const { return L1_SCALE_INDEX + ping_pong_idx; }

    __aicore__ inline uint32_t get_l0_slot(uint32_t ping_pong_idx) const { return L0_INDEX + ping_pong_idx; }

    // Main loop of matrix multiplication, including L1 prefetching, L1 to L0 copy, Mmad compute, and L0 to GM copy
    template <typename TensorA, typename TensorB, typename TensorAs, typename TensorBs, typename TensorC,
              typename TensorQuantScale, typename TensorQuantOffset>
    __aicore__ inline void process_loop(
        const TensorA& a, const TensorB& b, const TensorAs& as, const TensorBs& bs, TensorC& c,
        const TensorQuantScale& quant_scale, const TensorQuantOffset& quant_offset)
    {
        auto l1_layout_a = asc::te::make_frame_layout<asc::te::nz_layout_ptn, C0_ELEMENT_B4>(
            Trait::base_m, Trait::base_k * Trait::step_k);
        auto l1_layout_b = asc::te::make_frame_layout<asc::te::zn_layout_ptn, C0_ELEMENT_B4>(
            Trait::base_k * Trait::step_k, Trait::base_n);
        auto l1_layout_as = asc::te::make_frame_layout<asc::te::zz_layout_ptn, C0_ELEMENT_SCALE>(
            Trait::base_m, base_scale_k * Trait::step_k * Trait::scale_factor_k);
        auto l1_layout_bs = asc::te::make_frame_layout<asc::te::nn_layout_ptn, C0_ELEMENT_SCALE>(
            base_scale_k * Trait::step_k * Trait::scale_factor_k, Trait::base_n);

        // Malloc L1 and L0 buffers
        __cbuf__ fp4x2_e1m2_t l1_buf_a_ping[l1_size_a / 2];
        __cbuf__ fp4x2_e1m2_t l1_buf_a_pong[l1_size_a / 2];
        __cbuf__ fp4x2_e1m2_t l1_buf_b_ping[l1_size_b / 2];
        __cbuf__ fp4x2_e1m2_t l1_buf_b_pong[l1_size_b / 2];
        __cbuf__ fp8_e8m0_t l1_buf_as_ping[l1_size_as];
        __cbuf__ fp8_e8m0_t l1_buf_as_pong[l1_size_as];
        __cbuf__ fp8_e8m0_t l1_buf_bs_ping[l1_size_bs];
        __cbuf__ fp8_e8m0_t l1_buf_bs_pong[l1_size_bs];
        __ca__ fp4x2_e1m2_t l0_buf_a_ping[l0_size_a / 2];
        __ca__ fp4x2_e1m2_t l0_buf_a_pong[l0_size_a / 2];
        __cb__ fp4x2_e1m2_t l0_buf_b_ping[l0_size_b / 2];
        __cb__ fp4x2_e1m2_t l0_buf_b_pong[l0_size_b / 2];
        __cc__ float l0_buf_c[l0_size_c];

        // [Answer-2] 量化参数的 L1 缓冲区
        __cbuf__ uint64_t l1_buf_quant_scale[Trait::base_n];
        __cbuf__ uint64_t l1_buf_quant_offset[Trait::base_n];

        auto l0_ptr_a_ping = asc::te::make_mem_ptr(l0_buf_a_ping);
        auto l0_ptr_a_pong = asc::te::make_mem_ptr(l0_buf_a_pong);
        auto l0_ptr_b_ping = asc::te::make_mem_ptr(l0_buf_b_ping);
        auto l0_ptr_b_pong = asc::te::make_mem_ptr(l0_buf_b_pong);
        auto l0_ptr_as_ping = asc::te::make_mem_ptr<asc::te::location::l0scalea, fp8_e8m0_t>(
            reinterpret_cast<uint64_t>(l0_buf_a_ping) / 16);
        auto l0_ptr_as_pong = asc::te::make_mem_ptr<asc::te::location::l0scalea, fp8_e8m0_t>(
            reinterpret_cast<uint64_t>(l0_buf_a_pong) / 16);
        auto l0_ptr_bs_ping = asc::te::make_mem_ptr<asc::te::location::l0scaleb, fp8_e8m0_t>(
            reinterpret_cast<uint64_t>(l0_buf_b_ping) / 16);
        auto l0_ptr_bs_pong = asc::te::make_mem_ptr<asc::te::location::l0scaleb, fp8_e8m0_t>(
            reinterpret_cast<uint64_t>(l0_buf_b_pong) / 16);

        auto l1_tensor_a_ping = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_a_ping), l1_layout_a);
        auto l1_tensor_a_pong = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_a_pong), l1_layout_a);
        auto l1_tensor_b_ping = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_b_ping), l1_layout_b);
        auto l1_tensor_b_pong = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_b_pong), l1_layout_b);
        auto l1_tensor_as_ping = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_as_ping), l1_layout_as);
        auto l1_tensor_as_pong = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_as_pong), l1_layout_as);
        auto l1_tensor_bs_ping = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_bs_ping), l1_layout_bs);
        auto l1_tensor_bs_pong = asc::te::make_tensor(asc::te::make_mem_ptr(l1_buf_bs_pong), l1_layout_bs);

        // L1 quant tensors
        auto l1_quant_scale = asc::te::make_tensor(
            asc::te::make_mem_ptr(l1_buf_quant_scale),
            asc::te::make_frame_layout<asc::te::nd_layout_ptn, uint64_t>(1, Trait::base_n));
        auto l1_quant_offset = asc::te::make_tensor(
            asc::te::make_mem_ptr(l1_buf_quant_offset),
            asc::te::make_frame_layout<asc::te::nd_layout_ptn, uint64_t>(1, Trait::base_n));
        (void)l1_quant_offset; // 当前 4 参数 Copy 仅使用 scale，offset 预留待硬件 API 扩展

        // ============================================================
        // Mutex slot-lock synchronization: no preset needed (first asc_lock acquires a free slot).
        // ============================================================

        // ============================================================
        // N/M outer loops + K main loop
        // ============================================================
        for (uint32_t n_block_idx = 0; n_block_idx < n_loop_count; n_block_idx++) {
            uint16_t cur_n = (n_block_idx + 1 == n_loop_count) ? tail_n : Trait::base_n;
            for (uint32_t m_block_idx = 0; m_block_idx < m_loop_count; m_block_idx++) {
                uint16_t cur_m = (m_block_idx + 1 == m_loop_count) ? tail_m : Trait::base_m;
                // Reset the data_copy_in progress for each (m_block_idx, n_block_idx) tile.
                uint32_t data_next_k_chunk_idx = 0;  // K-direction chunk index for the next A/B data_copy_in.
                uint32_t scale_next_k_chunk_idx = 0; // K-direction chunk index for the next As/Bs data_copy_in.
                uint8_t data_copy_in_idx = 0;       // L1 buffer index written by the next A/B data_copy_in (0=Ping, 1=Pong).
                uint8_t scale_copy_in_idx = 0; // L1 buffer index written by the next As/Bs data_copy_in (0=Ping, 1=Pong).

                // Acquire L0C for Cube write (waits for Fixpipe to release it from the previous tile).
                asc_lock(PIPE_M, L0C_INDEX);

                // ---- Copy in the first chunk of A/B and As/Bs ----
                uint32_t data_write_slot = get_data_slot(data_copy_in_idx);
                asc_lock(PIPE_MTE2, data_write_slot); // A/B Ping writable.

                constexpr uint32_t step_cur_k = data_chunk_step * Trait::base_k;
                // A: GM -> L1
                asc::te::copy(
                    gm_to_l1_atom, l1_tensor_a_ping,
                    a.slice(
                        asc::te::make_coord(m_block_idx * Trait::base_m, data_next_k_chunk_idx * Trait::base_k),
                        asc::te::make_shape(cur_m, step_cur_k)));
                // B: GM -> L1
                asc::te::copy(
                    gm_to_l1_atom, l1_tensor_b_ping,
                    b.slice(
                        asc::te::make_coord(data_next_k_chunk_idx * Trait::base_k, n_block_idx * Trait::base_n),
                        asc::te::make_shape(step_cur_k, cur_n)));

                asc_unlock(PIPE_MTE2, data_write_slot); // A/B Ping written.
                data_next_k_chunk_idx += data_chunk_step;
                data_copy_in_idx ^= 1; // Write to Pong next time.

                uint32_t scale_write_slot = get_scale_slot(scale_copy_in_idx);
                asc_lock(PIPE_MTE2, scale_write_slot); // As/Bs Ping writable.

                constexpr uint32_t step_cur_scale_k =
                    align_even(AscendC::Std::ceil_div(scale_chunk_step * Trait::base_k, SCALE_CEIL_NUMBER));
                // scale_a: GM -> L1
                asc::te::copy(
                    gm_to_l1_atom, l1_tensor_as_ping,
                    as.slice(
                        asc::te::make_coord(m_block_idx * Trait::base_m, scale_next_k_chunk_idx * base_scale_k),
                        asc::te::make_shape(cur_m, step_cur_scale_k)));
                // scale_b: GM -> L1
                asc::te::copy(
                    gm_to_l1_atom, l1_tensor_bs_ping,
                    bs.slice(
                        asc::te::make_coord(scale_next_k_chunk_idx * base_scale_k, n_block_idx * Trait::base_n),
                        asc::te::make_shape(step_cur_scale_k, cur_n)));

                asc_unlock(PIPE_MTE2, scale_write_slot); // As/Bs Ping written.
                scale_next_k_chunk_idx += scale_chunk_step;
                scale_copy_in_idx ^= 1;

                auto l0_layout_c = asc::te::make_frame_layout<asc::te::nz_layout_ptn, C0_ELEMENT_L0C>(cur_m, cur_n);
                auto l0_tensor_c = asc::te::make_tensor(asc::te::make_mem_ptr(l0_buf_c), l0_layout_c);

                // ---- K-direction main loop ----
                for (uint32_t k_block_idx = 0; k_block_idx < k_loop_count; k_block_idx++) {
                    constexpr uint16_t cur_k = Trait::base_k;
                    // Determine the L1 read buffer (Ping/Pong) for the current k_block_idx.
                    uint32_t data_read_idx = (k_block_idx / data_chunk_step) % 2;
                    uint32_t scale_read_idx = (k_block_idx / scale_chunk_step) % 2;
                    uint32_t k_offset_in_data_chunk = k_block_idx % data_chunk_step;
                    uint32_t k_offset_in_scale_chunk = k_block_idx % scale_chunk_step;

                    const auto& l1_read_buf_a = (data_read_idx == 0) ? l1_tensor_a_ping : l1_tensor_a_pong;
                    const auto& l1_read_buf_b = (data_read_idx == 0) ? l1_tensor_b_ping : l1_tensor_b_pong;
                    const auto& l1_read_buf_as = (scale_read_idx == 0) ? l1_tensor_as_ping : l1_tensor_as_pong;
                    const auto& l1_read_buf_bs = (scale_read_idx == 0) ? l1_tensor_bs_ping : l1_tensor_bs_pong;

                    auto l0_layout_a = asc::te::make_frame_layout<asc::te::nz_layout_ptn, C0_ELEMENT_B4>(cur_m, cur_k);
                    auto l0_layout_b = asc::te::make_frame_layout<asc::te::zn_layout_ptn, C0_ELEMENT_B4>(cur_k, cur_n);
                    auto l0_tensor_a_ping = asc::te::make_tensor(l0_ptr_a_ping, l0_layout_a);
                    auto l0_tensor_a_pong = asc::te::make_tensor(l0_ptr_a_pong, l0_layout_a);
                    auto l0_tensor_b_ping = asc::te::make_tensor(l0_ptr_b_ping, l0_layout_b);
                    auto l0_tensor_b_pong = asc::te::make_tensor(l0_ptr_b_pong, l0_layout_b);

                    constexpr uint32_t cur_scale_k = align_even(AscendC::Std::ceil_div(cur_k, SCALE_CEIL_NUMBER));
                    auto l0_layout_as =
                        asc::te::make_frame_layout<asc::te::zz_layout_ptn, C0_ELEMENT_SCALE>(cur_m, cur_scale_k);
                    auto l0_layout_bs =
                        asc::te::make_frame_layout<asc::te::nn_layout_ptn, C0_ELEMENT_SCALE>(cur_scale_k, cur_n);
                    auto l0_tensor_as_ping = asc::te::make_tensor(l0_ptr_as_ping, l0_layout_as);
                    auto l0_tensor_as_pong = asc::te::make_tensor(l0_ptr_as_pong, l0_layout_as);
                    auto l0_tensor_bs_ping = asc::te::make_tensor(l0_ptr_bs_ping, l0_layout_bs);
                    auto l0_tensor_bs_pong = asc::te::make_tensor(l0_ptr_bs_pong, l0_layout_bs);

                    // Select the L0 double buffer.
                    const auto& l0_tensor_a = (mte1_db_flag == 0) ? l0_tensor_a_ping : l0_tensor_a_pong;
                    const auto& l0_tensor_b = (mte1_db_flag == 0) ? l0_tensor_b_ping : l0_tensor_b_pong;
                    const auto& l0_tensor_as = (mte1_db_flag == 0) ? l0_tensor_as_ping : l0_tensor_as_pong;
                    const auto& l0_tensor_bs = (mte1_db_flag == 0) ? l0_tensor_bs_ping : l0_tensor_bs_pong;

                    // ---- Reverse synchronization: wait for the previous Compute to release the L0 buffer ----
                    uint32_t l0_slot = get_l0_slot(mte1_db_flag);
                    asc_lock(PIPE_MTE1, l0_slot);

                    // ---- Forward synchronization ----
                    // Acquire the L1 chunk for reading at chunk start (waits for data_copy_in to finish writing it).
                    if (k_offset_in_data_chunk == 0) {
                        asc_lock(PIPE_MTE1, get_data_slot(data_read_idx));
                    }
                    if (k_offset_in_scale_chunk == 0) {
                        asc_lock(PIPE_MTE1, get_scale_slot(scale_read_idx));
                    }

                    // ---- data_load: L1 -> L0 ----
                    // A:L1 -> L0A
                    asc::te::copy(
                        l1_to_l0a_atom, l0_tensor_a,
                        l1_read_buf_a.slice(
                            asc::te::make_coord(0, k_offset_in_data_chunk * Trait::base_k),
                            asc::te::make_shape(cur_m, cur_k)));
                    // B:L1 -> L0B
                    asc::te::copy(
                        l1_to_l0b_atom, l0_tensor_b,
                        l1_read_buf_b.slice(
                            asc::te::make_coord(k_offset_in_data_chunk * Trait::base_k, 0),
                            asc::te::make_shape(cur_k, cur_n)));

                    // scale_a: L1 -> L0AScale
                    asc::te::copy(
                        l1_to_l0scalea_atom, l0_tensor_as,
                        l1_read_buf_as.slice(
                            asc::te::make_coord(0, k_offset_in_scale_chunk * base_scale_k),
                            asc::te::make_shape(cur_m, cur_scale_k)));
                    // scale_b: L1 -> L0BScale
                    asc::te::copy(
                        l1_to_l0scaleb_atom, l0_tensor_bs,
                        l1_read_buf_bs.slice(
                            asc::te::make_coord(k_offset_in_scale_chunk * base_scale_k, 0),
                            asc::te::make_shape(cur_scale_k, cur_n)));

                    // ---- Reverse synchronization ----
                    // The current L1 chunk has been consumed; release it so data_copy_in can overwrite.
                    if (((k_offset_in_data_chunk + 1) == data_chunk_step) || (k_block_idx + 1 == k_loop_count)) {
                        asc_unlock(PIPE_MTE1, get_data_slot(data_read_idx));
                    }
                    if (((k_offset_in_scale_chunk + 1) == scale_chunk_step) || (k_block_idx + 1 == k_loop_count)) {
                        asc_unlock(PIPE_MTE1, get_scale_slot(scale_read_idx));
                    }

                    // L0 written, ready for Cube.
                    asc_unlock(PIPE_MTE1, l0_slot);

                    // ---- Compute: Mmad matrix multiply-accumulate ----
                    asc_lock(PIPE_M, l0_slot);
                    asc::te::mmad_params params{cur_m, cur_n, cur_k, asc::te::unit_flag_mode::disable, true};
                    params.init_with_zero = (k_block_idx == 0);

                    asc::te::mmad(mmad_atom.with(params), l0_tensor_c, l0_tensor_a, l0_tensor_b);
                    // M_MTE1 reverse synchronization: release the L0 buffer for the next data_load.
                    asc_unlock(PIPE_M, l0_slot);
                    mte1_db_flag ^= 1;

                    // ---- Copy in the next L1 chunk so data_copy_in overlaps with compute in the pipeline ----
                    // Trigger conditions:
                    //   (1) k_block_idx == 0: when computing the first K block, A1/B1 Pong has not been used and can be
                    //   copied in directly. (2) The last base_k of the current L1 chunk has been consumed, so the buffer
                    //   can be overwritten.
                    if (((k_block_idx == 0) || ((k_offset_in_data_chunk + 1) == data_chunk_step)) &&
                        data_next_k_chunk_idx < k_loop_count) {
                        const auto& l1_write_buf_a = (data_copy_in_idx == 0) ? l1_tensor_a_ping : l1_tensor_a_pong;
                        const auto& l1_write_buf_b = (data_copy_in_idx == 0) ? l1_tensor_b_ping : l1_tensor_b_pong;
                        uint32_t data_write_slot = get_data_slot(data_copy_in_idx);
                        asc_lock(PIPE_MTE2, data_write_slot);
                        // A: GM -> L1
                        asc::te::copy(
                            gm_to_l1_atom, l1_write_buf_a,
                            a.slice(
                                asc::te::make_coord(m_block_idx * Trait::base_m, data_next_k_chunk_idx * Trait::base_k),
                                asc::te::make_shape(cur_m, step_cur_k)));
                        // B: GM -> L1
                        asc::te::copy(
                            gm_to_l1_atom, l1_write_buf_b,
                            b.slice(
                                asc::te::make_coord(data_next_k_chunk_idx * Trait::base_k, n_block_idx * Trait::base_n),
                                asc::te::make_shape(step_cur_k, cur_n)));

                        asc_unlock(PIPE_MTE2, data_write_slot);
                        data_next_k_chunk_idx += data_chunk_step;
                        data_copy_in_idx ^= 1;
                    }
                    if (((k_block_idx == 0) || ((k_offset_in_scale_chunk + 1) == scale_chunk_step)) &&
                        scale_next_k_chunk_idx < k_loop_count) {
                        const auto& l1_write_buf_as = (scale_copy_in_idx == 0) ? l1_tensor_as_ping : l1_tensor_as_pong;
                        const auto& l1_write_buf_bs = (scale_copy_in_idx == 0) ? l1_tensor_bs_ping : l1_tensor_bs_pong;
                        uint32_t scale_write_slot = get_scale_slot(scale_copy_in_idx);
                        asc_lock(PIPE_MTE2, scale_write_slot);
                        // scale_a: GM -> L1
                        asc::te::copy(
                            gm_to_l1_atom, l1_write_buf_as,
                            as.slice(
                                asc::te::make_coord(m_block_idx * Trait::base_m, scale_next_k_chunk_idx * base_scale_k),
                                asc::te::make_shape(cur_m, step_cur_scale_k)));
                        // scale_b: GM -> L1
                        asc::te::copy(
                            gm_to_l1_atom, l1_write_buf_bs,
                            bs.slice(
                                asc::te::make_coord(scale_next_k_chunk_idx * base_scale_k, n_block_idx * Trait::base_n),
                                asc::te::make_shape(step_cur_scale_k, cur_n)));

                        asc_unlock(PIPE_MTE2, scale_write_slot);
                        scale_next_k_chunk_idx += scale_chunk_step;
                        scale_copy_in_idx ^= 1;
                    }
                }
                // ---- copy quant params: GM -> L1 ----
                asc_lock(PIPE_MTE2, QUANT_INDEX);
                // [Answer-3] 将 quant_scale 从 GM 拷贝到 L1
                asc::te::copy(
                    gm_to_l1_atom, l1_quant_scale,
                    quant_scale.slice(
                        asc::te::make_coord(0, n_block_idx * Trait::base_n),
                        asc::te::make_shape(1, cur_n)));
                asc_unlock(PIPE_MTE2, QUANT_INDEX);

                // ---- copy_out: L0C -> GM (with quantization) ----
                asc_unlock(PIPE_M, L0C_INDEX);
                asc_lock(PIPE_FIX, QUANT_INDEX);
                asc_lock(PIPE_FIX, L0C_INDEX);
                asc::te::l0c_to_gm_params fixpipe_params;
                // [Answer-4] 4 参数 Copy：附加量化张量 l1_quant_scale
                asc::te::copy(
                    l0c_to_gm_atom.with(fixpipe_params),
                    c.slice(
                        asc::te::make_coord(m_block_idx * Trait::base_m, n_block_idx * Trait::base_n),
                        asc::te::make_shape(cur_m, cur_n)),
                    l0_tensor_c, l1_quant_scale);
                asc_unlock(PIPE_FIX, L0C_INDEX);
                asc_unlock(PIPE_FIX, QUANT_INDEX);
            }
        }
    }

    __aicore__ inline void init_compute_params()
    {
        // ---- 1. Compute current-core M/N iteration indexes and GM start offset ----
        constexpr uint32_t m_iter = AscendC::Std::ceil_div(Trait::M, Trait::single_core_m);
        m_iter_idx = block_idx % m_iter;
        n_iter_idx = block_idx / m_iter;

        // ---- 2. Compute actual M/N dimensions of the current core ----
        // The last block may be smaller than single_core.
        actual_single_core_m = Trait::M - m_iter_idx * Trait::single_core_m;
        actual_single_core_m = actual_single_core_m < Trait::single_core_m ? actual_single_core_m : Trait::single_core_m;
        actual_single_core_n = Trait::N - n_iter_idx * Trait::single_core_n;
        actual_single_core_n = actual_single_core_n < Trait::single_core_n ? actual_single_core_n : Trait::single_core_n;

        // ---- 3. Compute the loop counts for the M/N/K dimensions ----
        m_loop_count = AscendC::Std::ceil_div(actual_single_core_m, Trait::base_m);
        n_loop_count = AscendC::Std::ceil_div(actual_single_core_n, Trait::base_n);
        k_loop_count = AscendC::Std::ceil_div(Trait::single_core_k, Trait::base_k);

        // ---- 4. Compute the M/N-direction tiling parameters ----
        tail_m = (actual_single_core_m % Trait::base_m != 0) ? actual_single_core_m % Trait::base_m : Trait::base_m;
        tail_n = (actual_single_core_n % Trait::base_n != 0) ? actual_single_core_n % Trait::base_n : Trait::base_n;
    }

private:
    uint32_t actual_single_core_m, actual_single_core_n;
    uint32_t m_iter_idx, n_iter_idx;
    uint32_t m_loop_count, n_loop_count, k_loop_count;
    uint32_t tail_m, tail_n;
    uint8_t mte1_db_flag = 0;

    static constexpr uint32_t scale_k = align_even(AscendC::Std::ceil_div(Trait::K, SCALE_CEIL_NUMBER));
    static constexpr uint32_t base_scale_k = align_even(AscendC::Std::ceil_div(Trait::base_k, SCALE_CEIL_NUMBER));

    static constexpr size_t l1_size_a = Trait::base_m * (Trait::base_k * Trait::step_k);
    static constexpr size_t l1_size_b = (Trait::base_k * Trait::step_k) * Trait::base_n;
    static constexpr size_t l1_size_as = Trait::base_m * (base_scale_k * Trait::step_k * Trait::scale_factor_k);
    static constexpr size_t l1_size_bs = (base_scale_k * Trait::step_k * Trait::scale_factor_k) * Trait::base_n;

    static constexpr size_t l0_size_a = Trait::base_m * Trait::base_k;
    static constexpr size_t l0_size_b = Trait::base_k * Trait::base_n;
    static constexpr size_t l0_size_c = Trait::base_m * Trait::base_n;

    static constexpr auto gm_to_l1_atom = asc::te::make_copy(asc::te::copy_gm_to_l1{});
    static constexpr auto l1_to_l0a_atom = asc::te::make_copy(asc::te::copy_l1_to_l0a{});
    static constexpr auto l1_to_l0b_atom = asc::te::make_copy(asc::te::copy_l1_to_l0b{});
    static constexpr auto l1_to_l0scalea_atom = asc::te::make_copy(asc::te::copy_l1_to_l0scalea{});
    static constexpr auto l1_to_l0scaleb_atom = asc::te::make_copy(asc::te::copy_l1_to_l0scaleb{});
    static constexpr auto l0c_to_gm_atom = asc::te::make_copy(asc::te::copy_l0c_to_gm{});
    static constexpr auto mmad_atom = asc::te::make_mmad(asc::te::mmad_operation{}, mmad_trait_mx{});

    static constexpr uint32_t data_chunk_step = Trait::step_k;
    static constexpr uint32_t scale_chunk_step = Trait::step_k * Trait::scale_factor_k;
};

template <typename Trait>
__global__ __cube__ void mmadmx_quant_custom(
    __gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* as, __gm__ uint8_t* bs, __gm__ uint8_t* c,
    __gm__ uint8_t* quant_scale, __gm__ uint8_t* quant_offset)
{
    asc_init();
    kernel_mmadmx_quant<Trait> op;
    op.process(
        reinterpret_cast<__gm__ fp4x2_e1m2_t*>(a), reinterpret_cast<__gm__ fp4x2_e1m2_t*>(b),
        reinterpret_cast<__gm__ fp8_e8m0_t*>(as), reinterpret_cast<__gm__ fp8_e8m0_t*>(bs),
        reinterpret_cast<__gm__ int8_t*>(c), reinterpret_cast<__gm__ uint64_t*>(quant_scale),
        reinterpret_cast<__gm__ uint64_t*>(quant_offset));

    asc_sync_pipe(PIPE_ALL);
}

#endif


In [ ]:
%%writefile src/07_08_tensor_api_mxfp4_matmul/mmad_mx_quant_answer.asc
/**
 * Copyright (c) 2026 Huawei Technologies Co., Ltd.
 * This program is free software, you can redistribute it and/or modify it under the terms and conditions of
 * CANN Open Software License Agreement Version 2.0 (the "License").
 * Please refer to the License for details. You may not use this file except in compliance with the License.
 * THIS FILE IS PROVIDED ON AN "AS IS" BASIS, WITHOUT WARRANTIES OF ANY KIND, EITHER EXPRESS OR IMPLIED,
 * INCLUDING BUT NOT LIMITED TO NON-INFRINGEMENT, MERCHANTABILITY, OR FITNESS FOR A PARTICULAR PURPOSE.
 * See LICENSE in the root of the software repository for the full text of the License.
 */

#include "mmad_mx_host.h"
#include "mmad_mx_quant_answer_kernel.h"

namespace quant_answer_config {
using Trait = kernel_trait<8192, 8192, 8192, 2048, 1024, 8192, 256, 256, 256, 2, 4>;
constexpr uint32_t NUM_BLOCKS = 32;

constexpr uint32_t scale_k_unaligned = (Trait::K + SCALE_CEIL_NUMBER - 1) / SCALE_CEIL_NUMBER;
constexpr uint32_t s_k = ((scale_k_unaligned + SCALE_ALIGN_NUMBER - 1) / SCALE_ALIGN_NUMBER) * SCALE_ALIGN_NUMBER;
}

int32_t main(int32_t argc, char *argv[])
{
    (void)argc;
    (void)argv;
    using namespace quant_answer_config;

    constexpr size_t a_file_size = static_cast<size_t>(Trait::M * Trait::K) * sizeof(uint8_t) / 2;
    constexpr size_t b_file_size = static_cast<size_t>(Trait::N * Trait::K) * sizeof(uint8_t) / 2;
    constexpr size_t as_file_size = static_cast<size_t>(Trait::M * s_k) * sizeof(uint8_t);
    constexpr size_t bs_file_size = static_cast<size_t>(Trait::N * s_k) * sizeof(uint8_t);
    // 输出为 INT8，每个元素 1 字节
    constexpr size_t c_file_size = Trait::M * Trait::N * sizeof(int8_t);
    // 量化参数 shape [1, N]，类型 uint64_t
    constexpr size_t quant_file_size = Trait::N * sizeof(uint64_t);

    CHECK_ACL_RET(aclInit(nullptr));
    int32_t device_id = 0;
    CHECK_ACL_RET(aclrtSetDevice(device_id));

    aclrtStream stream = nullptr;
    CHECK_ACL_RET(aclrtCreateStream(&stream));

    uint8_t *a_host = nullptr, *b_host = nullptr, *as_host = nullptr, *bs_host = nullptr, *c_host = nullptr;
    uint8_t *a_device = nullptr, *b_device = nullptr, *as_device = nullptr, *bs_device = nullptr, *c_device = nullptr;
    uint8_t *quant_scale_host = nullptr, *quant_offset_host = nullptr;
    uint8_t *quant_scale_device = nullptr, *quant_offset_device = nullptr;

    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&a_host), a_file_size));
    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&b_host), b_file_size));
    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&as_host), as_file_size));
    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&bs_host), bs_file_size));
    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&c_host), c_file_size));
    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&quant_scale_host), quant_file_size));
    CHECK_ACL_RET(aclrtMallocHost(reinterpret_cast<void **>(&quant_offset_host), quant_file_size));

    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&a_device), a_file_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&b_device), b_file_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&as_device), as_file_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&bs_device), bs_file_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&c_device), c_file_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&quant_scale_device), quant_file_size, ACL_MEM_MALLOC_HUGE_FIRST));
    CHECK_ACL_RET(aclrtMalloc(reinterpret_cast<void **>(&quant_offset_device), quant_file_size, ACL_MEM_MALLOC_HUGE_FIRST));

    size_t file_size = a_file_size;
    if (!ReadFile("./input/x1_gm.bin", file_size, a_host, a_file_size)) { return 1; }
    file_size = b_file_size;
    if (!ReadFile("./input/x2_gm.bin", file_size, b_host, b_file_size)) { return 1; }
    file_size = as_file_size;
    if (!ReadFile("./input/x1_scale_gm.bin", file_size, as_host, as_file_size)) { return 1; }
    file_size = bs_file_size;
    if (!ReadFile("./input/x2_scale_gm.bin", file_size, bs_host, bs_file_size)) { return 1; }
    file_size = quant_file_size;
    if (!ReadFile("./input/quant_scale.bin", file_size, quant_scale_host, quant_file_size)) { return 1; }
    file_size = quant_file_size;
    if (!ReadFile("./input/quant_offset.bin", file_size, quant_offset_host, quant_file_size)) { return 1; }

    CHECK_ACL_RET(aclrtMemcpy(a_device, a_file_size, a_host, a_file_size, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL_RET(aclrtMemcpy(b_device, b_file_size, b_host, b_file_size, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL_RET(aclrtMemcpy(as_device, as_file_size, as_host, as_file_size, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL_RET(aclrtMemcpy(bs_device, bs_file_size, bs_host, bs_file_size, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL_RET(aclrtMemcpy(quant_scale_device, quant_file_size, quant_scale_host, quant_file_size, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL_RET(aclrtMemcpy(quant_offset_device, quant_file_size, quant_offset_host, quant_file_size, ACL_MEMCPY_HOST_TO_DEVICE));

    mmadmx_quant_custom<Trait><<<NUM_BLOCKS, 0, stream>>>(
        a_device, b_device, as_device, bs_device, c_device, quant_scale_device, quant_offset_device);
    CHECK_ACL_RET(aclrtSynchronizeStream(stream));

    CHECK_ACL_RET(aclrtMemcpy(c_host, c_file_size, c_device, c_file_size, ACL_MEMCPY_DEVICE_TO_HOST));
    if (!WriteFile("./output/quant_answer.bin", c_host, c_file_size)) { return 1; }

    (void)aclrtFree(a_device); (void)aclrtFree(b_device);
    (void)aclrtFree(as_device); (void)aclrtFree(bs_device);
    (void)aclrtFree(c_device);
    (void)aclrtFree(quant_scale_device); (void)aclrtFree(quant_offset_device);
    (void)aclrtFreeHost(a_host); (void)aclrtFreeHost(b_host);
    (void)aclrtFreeHost(as_host); (void)aclrtFreeHost(bs_host);
    (void)aclrtFreeHost(c_host);
    (void)aclrtFreeHost(quant_scale_host); (void)aclrtFreeHost(quant_offset_host);
    (void)aclrtDestroyStream(stream);
    (void)aclrtResetDevice(device_id);
    (void)aclFinalize();
    return 0;
}


In [ ]:
!cd src/07_08_tensor_api_mxfp4_matmul && bash run.sh --case=quant_answer

### 参考答案

参考实现的完整代码已写入 `mmad_mx_quant_answer_kernel.h`。课后练习和实践的详细答案见 `answer/07_08_tensor_api_mxfp4_matmul/answer.md`。